# Notebook 06: Gold-Standard Annotation

**Author:** Anthony Amit Biswas

## What this notebook does

Creates and validates the reproducible 70-note annotation sample and structured reference-standard workflow used for evaluation.


In [ ]:
# 1. Mount Google Drive

from google.colab import drive
from pathlib import Path

MOUNT_POINT = "/content/drive"

drive.mount(
    MOUNT_POINT,
    force_remount=True,
)

DRIVE_ROOT = Path(MOUNT_POINT) / "MyDrive"

print("-" * 80)
print("GOOGLE DRIVE STATUS")
print("-" * 80)
print(f"Drive root : {DRIVE_ROOT}")
print(f"Exists     : {DRIVE_ROOT.exists()}")
print(f"Is folder  : {DRIVE_ROOT.is_dir()}")

if not DRIVE_ROOT.is_dir():
    raise RuntimeError(
        "Google Drive was mounted, but MyDrive is not accessible."
    )

print("\nGoogle Drive mounted successfully.")

**Locating the actual dissertation files**

In [ ]:
# 2. Discovering Existing Dissertation Outputs

from pathlib import Path
import pandas as pd

TARGET_FILENAMES = [
    "icu_discharge_summary_cohort.csv.gz",
    "icu_discharge_summaries_preprocessed.csv.gz",
]

TARGET_KEYWORDS = [
    "medspacy",
    "scispacy",
    "discharge",
    "cohort",
    "preprocess",
]

print("-" * 100)
print("SEARCHING GOOGLE DRIVE FOR DISSERTATION OUTPUTS")
print("-" * 100)

# Exact searches for the two core cohort files
exact_matches = []

for filename in TARGET_FILENAMES:
    matches = list(DRIVE_ROOT.rglob(filename))

    print(f"\nExact filename: {filename}")
    print(f"Matches found : {len(matches)}")

    for path in matches:
        print(f"  {path}")

    exact_matches.extend(matches)

# Broader inventory of relevant files and folders
relevant_items = []

for path in DRIVE_ROOT.rglob("*"):
    name_lower = path.name.lower()

    if any(keyword in name_lower for keyword in TARGET_KEYWORDS):
        try:
            size_mb = (
                round(path.stat().st_size / (1024 ** 2), 3)
                if path.is_file()
                else None
            )
        except OSError:
            size_mb = None

        relevant_items.append(
            {
                "name": path.name,
                "type": "directory" if path.is_dir() else "file",
                "size_mb": size_mb,
                "full_path": str(path),
            }
        )

relevant_inventory_df = pd.DataFrame(relevant_items)

print("-" * 100)
print("RELEVANT FILE AND DIRECTORY INVENTORY")
print("-" * 100)
print(f"Relevant items found: {len(relevant_inventory_df):,}")

if not relevant_inventory_df.empty:
    relevant_inventory_df = (
        relevant_inventory_df
        .sort_values(
            ["type", "name", "full_path"],
            key=lambda column: column.astype(str).str.lower(),
        )
        .reset_index(drop=True)
    )

    display(relevant_inventory_df)
else:
    print("No relevant files or directories were found.")

**Resolving only the two core cohort files**

In [ ]:
# 3. Resolving Core Cohort Files

def resolve_unique_file(root, filename):
    """
    Resolve one exact filename anywhere below the supplied root.
    """

    matches = list(root.rglob(filename))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find '{filename}' anywhere under:\n{root}"
        )

    if len(matches) > 1:
        print(
            f"Warning: {len(matches)} copies of '{filename}' were found."
        )

        for index, path in enumerate(matches, start=1):
            print(f"{index}. {path}")

        # Prefer a copy stored inside a folder named outputs.
        output_matches = [
            path
            for path in matches
            if "outputs" in {
                part.lower()
                for part in path.parts
            }
        ]

        if len(output_matches) == 1:
            selected = output_matches[0]
        else:
            selected = max(
                matches,
                key=lambda path: path.stat().st_size,
            )
    else:
        selected = matches[0]

    return selected


ORIGINAL_COHORT_FILE = resolve_unique_file(
    DRIVE_ROOT,
    "icu_discharge_summary_cohort.csv.gz",
)

PREPROCESSED_FILE = resolve_unique_file(
    DRIVE_ROOT,
    "icu_discharge_summaries_preprocessed.csv.gz",
)

PROJECT_OUTPUT_DIRECTORY = ORIGINAL_COHORT_FILE.parent

ANNOTATION_DIRECTORY = (
    PROJECT_OUTPUT_DIRECTORY /
    "gold_standard_annotation"
)

ANNOTATION_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

resolved_files_df = pd.DataFrame(
    [
        {
            "resource": "Original ICU cohort",
            "path": str(ORIGINAL_COHORT_FILE),
            "size_mb": round(
                ORIGINAL_COHORT_FILE.stat().st_size / (1024 ** 2),
                2,
            ),
        },
        {
            "resource": "Preprocessed ICU cohort",
            "path": str(PREPROCESSED_FILE),
            "size_mb": round(
                PREPROCESSED_FILE.stat().st_size / (1024 ** 2),
                2,
            ),
        },
        {
            "resource": "Annotation output directory",
            "path": str(ANNOTATION_DIRECTORY),
            "size_mb": None,
        },
    ]
)

display(resolved_files_df)

print("\nCore cohort files resolved successfully.")

Inspecting cohort schemas

In [ ]:
# 4. Inspect Original and Preprocessed Cohort Schemas

import pandas as pd

original_preview_df = pd.read_csv(
    ORIGINAL_COHORT_FILE,
    nrows=5,
    low_memory=False,
)

preprocessed_preview_df = pd.read_csv(
    PREPROCESSED_FILE,
    nrows=5,
    low_memory=False,
)

print("-" * 100)
print("ORIGINAL ICU COHORT")
print("-" * 100)

print("Columns:")
print(original_preview_df.columns.tolist())

print("\nData types:")
display(original_preview_df.dtypes.rename("dtype").to_frame())

print("\nPreview:")
display(original_preview_df)

print("\n" + "-" * 100)
print("PREPROCESSED ICU COHORT")
print("-" * 100)

print("Columns:")
print(preprocessed_preview_df.columns.tolist())

print("\nData types:")
display(preprocessed_preview_df.dtypes.rename("dtype").to_frame())

print("\nPreview:")
display(preprocessed_preview_df)

**Resolving the required cohort columns**

In [ ]:
# 5. Resolving Required Cohort Columns

def resolve_column(
    columns,
    candidate_names,
    required=True,
):
    """
    Return the first column matching one of the supplied candidate names.
    Matching is case-insensitive.
    """

    lookup = {
        str(column).strip().lower(): column
        for column in columns
    }

    for candidate in candidate_names:
        normalized_candidate = candidate.strip().lower()

        if normalized_candidate in lookup:
            return lookup[normalized_candidate]

    if required:
        raise KeyError(
            "Required column could not be resolved.\n"
            f"Candidates: {candidate_names}\n"
            f"Available columns: {list(columns)}"
        )

    return None


ORIGINAL_NOTE_ID_COLUMN = resolve_column(
    original_preview_df.columns,
    ["note_id", "noteid"],
)

ORIGINAL_SUBJECT_ID_COLUMN = resolve_column(
    original_preview_df.columns,
    ["subject_id", "patient_id"],
)

ORIGINAL_HADM_ID_COLUMN = resolve_column(
    original_preview_df.columns,
    ["hadm_id", "admission_id"],
    required=False,
)

ORIGINAL_TEXT_COLUMN = resolve_column(
    original_preview_df.columns,
    [
        "text",
        "note_text",
        "raw_text",
        "discharge_summary",
    ],
)

PREPROCESSED_NOTE_ID_COLUMN = resolve_column(
    preprocessed_preview_df.columns,
    ["note_id", "noteid"],
)

PREPROCESSED_TEXT_COLUMN = resolve_column(
    preprocessed_preview_df.columns,
    [
        "preprocessed_text",
        "cleaned_text",
        "processed_text",
        "text",
    ],
)

resolved_cohort_columns_df = pd.DataFrame(
    [
        {
            "dataset": "Original cohort",
            "role": "Note identifier",
            "resolved_column": ORIGINAL_NOTE_ID_COLUMN,
        },
        {
            "dataset": "Original cohort",
            "role": "Patient identifier",
            "resolved_column": ORIGINAL_SUBJECT_ID_COLUMN,
        },
        {
            "dataset": "Original cohort",
            "role": "Admission identifier",
            "resolved_column": ORIGINAL_HADM_ID_COLUMN,
        },
        {
            "dataset": "Original cohort",
            "role": "Original note text",
            "resolved_column": ORIGINAL_TEXT_COLUMN,
        },
        {
            "dataset": "Preprocessed cohort",
            "role": "Note identifier",
            "resolved_column": PREPROCESSED_NOTE_ID_COLUMN,
        },
        {
            "dataset": "Preprocessed cohort",
            "role": "Preprocessed note text",
            "resolved_column": PREPROCESSED_TEXT_COLUMN,
        },
    ]
)

display(resolved_cohort_columns_df)

print("\nRequired cohort columns resolved successfully.")

**Discovering Notebook 05 outputs**

In [ ]:
# 6. Discovering medSpaCy Output Files

MEDSPACY_DIRECTORY = (
    PROJECT_OUTPUT_DIRECTORY /
    "medspacy"
)

if not MEDSPACY_DIRECTORY.is_dir():
    raise FileNotFoundError(
        f"medSpaCy output directory not found:\n{MEDSPACY_DIRECTORY}"
    )

medspacy_files = sorted(
    [
        path
        for path in MEDSPACY_DIRECTORY.rglob("*")
        if path.is_file()
    ],
    key=lambda path: str(path).lower(),
)

medspacy_inventory = []

for path in medspacy_files:
    medspacy_inventory.append(
        {
            "filename": path.name,
            "relative_path": str(
                path.relative_to(MEDSPACY_DIRECTORY)
            ),
            "suffixes": "".join(path.suffixes),
            "size_mb": round(
                path.stat().st_size / (1024 ** 2),
                4,
            ),
        }
    )

medspacy_inventory_df = pd.DataFrame(
    medspacy_inventory
)

print("-" * 100)
print("MEDSPACY OUTPUT INVENTORY")
print("-" * 100)
print(f"Files found: {len(medspacy_inventory_df):,}")

display(medspacy_inventory_df)

**Classifying the medSpaCy files**

In [ ]:
# 7. Classifying medSpaCy Outputs

def classify_medspacy_output(filename):
    """
    Infer the likely purpose of a medSpaCy file from its filename.
    """

    name = filename.lower()

    if "mention" in name or "entity" in name:
        return "mention_level"

    if (
        "note" in name
        and "category" in name
    ):
        return "note_category"

    if (
        "note" in name
        and "summary" in name
    ):
        return "note_summary"

    if (
        "patient" in name
        and "category" in name
    ):
        return "patient_category"

    if (
        "patient" in name
        and "summary" in name
    ):
        return "patient_summary"

    if "prevalence" in name:
        return "category_prevalence"

    if "qa" in name:
        return "quality_assurance"

    if (
        "manifest" in name
        or "config" in name
        or "validation" in name
    ):
        return "configuration_or_validation"

    return "other"


medspacy_inventory_df["classification"] = (
    medspacy_inventory_df["filename"].apply(
        classify_medspacy_output
    )
)

display(
    medspacy_inventory_df[
        [
            "filename",
            "classification",
            "size_mb",
            "relative_path",
        ]
    ].sort_values(
        [
            "classification",
            "filename",
        ]
    ).reset_index(drop=True)
)

print("\nClassification summary:")

display(
    medspacy_inventory_df[
        "classification"
    ]
    .value_counts(dropna=False)
    .rename_axis("classification")
    .reset_index(name="file_count")
)

**Resolving the main medSpaCy files**

In [ ]:
# 8. Resolving Principal medSpaCy Output Files

def select_best_file(
    inventory_df,
    classification,
):
    """
    Select the most likely principal file for a classification.

    Preference:
    1. compressed CSV;
    2. ordinary CSV;
    3. largest matching file.
    """

    candidates = inventory_df.loc[
        inventory_df["classification"] == classification
    ].copy()

    if candidates.empty:
        return None

    candidates["is_csv_gz"] = (
        candidates["filename"]
        .str.lower()
        .str.endswith(".csv.gz")
    )

    candidates["is_csv"] = (
        candidates["filename"]
        .str.lower()
        .str.endswith(".csv")
    )

    candidates = candidates.sort_values(
        [
            "is_csv_gz",
            "is_csv",
            "size_mb",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )

    selected_relative_path = (
        candidates.iloc[0]["relative_path"]
    )

    return (
        MEDSPACY_DIRECTORY /
        selected_relative_path
    )


MEDSPACY_MENTION_FILE = select_best_file(
    medspacy_inventory_df,
    "mention_level",
)

MEDSPACY_NOTE_CATEGORY_FILE = select_best_file(
    medspacy_inventory_df,
    "note_category",
)

MEDSPACY_NOTE_SUMMARY_FILE = select_best_file(
    medspacy_inventory_df,
    "note_summary",
)

MEDSPACY_PREVALENCE_FILE = select_best_file(
    medspacy_inventory_df,
    "category_prevalence",
)

resolved_medspacy_df = pd.DataFrame(
    [
        {
            "purpose": "Mention-level output",
            "path": (
                str(MEDSPACY_MENTION_FILE)
                if MEDSPACY_MENTION_FILE
                else None
            ),
        },
        {
            "purpose": "Note-category output",
            "path": (
                str(MEDSPACY_NOTE_CATEGORY_FILE)
                if MEDSPACY_NOTE_CATEGORY_FILE
                else None
            ),
        },
        {
            "purpose": "Note-level summary",
            "path": (
                str(MEDSPACY_NOTE_SUMMARY_FILE)
                if MEDSPACY_NOTE_SUMMARY_FILE
                else None
            ),
        },
        {
            "purpose": "Category prevalence",
            "path": (
                str(MEDSPACY_PREVALENCE_FILE)
                if MEDSPACY_PREVALENCE_FILE
                else None
            ),
        },
    ]
)

display(resolved_medspacy_df)

**Inspecting resolved medSpaCy schemas**

In [ ]:
# 9. Inspecting Resolved medSpaCy Schemas

resolved_medspacy_files = {
    "Mention-level output": MEDSPACY_MENTION_FILE,
    "Note-category output": MEDSPACY_NOTE_CATEGORY_FILE,
    "Note-level summary": MEDSPACY_NOTE_SUMMARY_FILE,
    "Category prevalence": MEDSPACY_PREVALENCE_FILE,
}

for label, path in resolved_medspacy_files.items():

    print("\n" + "-" * 100)
    print(label.upper())
    print("-" * 100)

    if path is None:
        print("No matching file was resolved.")
        continue

    print(f"File: {path}")

    try:
        preview_df = pd.read_csv(
            path,
            nrows=5,
            low_memory=False,
        )

        print(f"Columns ({len(preview_df.columns)}):")
        print(preview_df.columns.tolist())

        display(preview_df)

    except Exception as error:
        print(f"Could not inspect file: {error}")

**Correct and lock the required files**

In [ ]:
# 10. Lock Confirmed Files and Columns

PREPROCESSED_TEXT_COLUMN = "clean_text"

MEDSPACY_NOTE_CATEGORY_FILE = (
    MEDSPACY_DIRECTORY
    / "summary_outputs"
    / "medspacy_note_category_summary.csv.gz"
)

MEDSPACY_NOTE_SUMMARY_FILE = (
    MEDSPACY_DIRECTORY
    / "summary_outputs"
    / "medspacy_note_summary.csv.gz"
)

MEDSPACY_PREVALENCE_FILE = (
    MEDSPACY_DIRECTORY
    / "summary_outputs"
    / "medspacy_category_prevalence.csv"
)

confirmed_resources = {
    "Original cohort": ORIGINAL_COHORT_FILE,
    "Preprocessed cohort": PREPROCESSED_FILE,
    "medSpaCy note-category summary": MEDSPACY_NOTE_CATEGORY_FILE,
    "medSpaCy note summary": MEDSPACY_NOTE_SUMMARY_FILE,
    "medSpaCy category prevalence": MEDSPACY_PREVALENCE_FILE,
}

resource_validation = []

for resource, path in confirmed_resources.items():
    resource_validation.append(
        {
            "resource": resource,
            "path": str(path),
            "exists": path.is_file(),
            "size_mb": (
                round(path.stat().st_size / (1024 ** 2), 3)
                if path.is_file()
                else None
            ),
        }
    )

resource_validation_df = pd.DataFrame(resource_validation)

display(resource_validation_df)

missing_resources = resource_validation_df.loc[
    ~resource_validation_df["exists"],
    "resource",
].tolist()

if missing_resources:
    raise FileNotFoundError(
        "Required resources are missing:\n- "
        + "\n- ".join(missing_resources)
    )

print("\nAll required Notebook 06 resources validated.")

**Loading only the required data**

In [ ]:
# 11. Loading Required Cohort and medSpaCy Data

cohort_columns = [
    "note_id",
    "subject_id",
    "hadm_id",
    "text",
    "word_count",
]

cohort_df = pd.read_csv(
    ORIGINAL_COHORT_FILE,
    usecols=cohort_columns,
    low_memory=False,
)

note_summary_df = pd.read_csv(
    MEDSPACY_NOTE_SUMMARY_FILE,
    low_memory=False,
)

note_category_df = pd.read_csv(
    MEDSPACY_NOTE_CATEGORY_FILE,
    low_memory=False,
)

category_prevalence_df = pd.read_csv(
    MEDSPACY_PREVALENCE_FILE,
    low_memory=False,
)

# Standardise key identifiers.
for dataframe in [
    cohort_df,
    note_summary_df,
    note_category_df,
]:
    dataframe["note_id"] = dataframe["note_id"].astype(str)
    dataframe["subject_id"] = dataframe["subject_id"].astype(str)
    dataframe["hadm_id"] = dataframe["hadm_id"].astype(str)

cohort_df["text"] = cohort_df["text"].fillna("").astype(str)

print("-" * 90)
print("LOADED DATASETS")
print("-" * 90)

print(f"Original cohort rows       : {len(cohort_df):,}")
print(f"Note-summary rows          : {len(note_summary_df):,}")
print(f"Note-category rows         : {len(note_category_df):,}")
print(f"Complication categories    : {len(category_prevalence_df):,}")

**Validating dataset relationships**

In [ ]:
# 12. Validating Dataset Relationships

validation_results = {
    "cohort_has_65323_rows":
        len(cohort_df) == 65_323,

    "cohort_note_ids_unique":
        cohort_df["note_id"].is_unique,

    "note_summary_has_65323_rows":
        len(note_summary_df) == 65_323,

    "note_summary_note_ids_unique":
        note_summary_df["note_id"].is_unique,

    "all_summary_notes_in_cohort":
        set(note_summary_df["note_id"]).issubset(
            set(cohort_df["note_id"])
        ),

    "all_category_notes_in_cohort":
        set(note_category_df["note_id"]).issubset(
            set(cohort_df["note_id"])
        ),

    "no_empty_original_text":
        cohort_df["text"].str.strip().ne("").all(),
}

validation_df = pd.DataFrame(
    [
        {
            "validation_check": check,
            "passed": bool(result),
        }
        for check, result in validation_results.items()
    ]
)

display(validation_df)

failed_checks = validation_df.loc[
    ~validation_df["passed"],
    "validation_check",
].tolist()

if failed_checks:
    raise ValueError(
        "Dataset validation failed:\n- "
        + "\n- ".join(failed_checks)
    )

print("\nCohort and medSpaCy outputs align successfully.")

**Creating the sampling dataset**

I have used:

original, unmodified note text for manual annotation;
medSpaCy outputs only to support stratified sampling;
one note per patient to reduce patient-level dependence.

In [ ]:
# 13. Constructing Annotation Sampling Dataset

sampling_df = cohort_df.merge(
    note_summary_df[
        [
            "note_id",
            "total_complication_mentions",
            "affirmed_complication_mentions",
            "unique_complication_categories",
            "categories_with_affirmed_mentions",
            "has_any_complication_mention",
            "has_current_affirmed_complication",
        ]
    ],
    on="note_id",
    how="left",
    validate="one_to_one",
)

summary_columns = [
    "total_complication_mentions",
    "affirmed_complication_mentions",
    "unique_complication_categories",
    "categories_with_affirmed_mentions",
]

sampling_df[summary_columns] = (
    sampling_df[summary_columns]
    .fillna(0)
    .astype(int)
)

boolean_columns = [
    "has_any_complication_mention",
    "has_current_affirmed_complication",
]

sampling_df[boolean_columns] = (
    sampling_df[boolean_columns]
    .fillna(False)
    .astype(bool)
)

sampling_df["text_character_count"] = (
    sampling_df["text"].str.len()
)

print("-" * 90)
print("ANNOTATION SAMPLING DATASET")
print("-" * 90)

print(f"Rows                     : {len(sampling_df):,}")
print(f"Unique notes             : {sampling_df['note_id'].nunique():,}")
print(f"Unique patients          : {sampling_df['subject_id'].nunique():,}")
print(
    "Notes with affirmed complications: "
    f"{sampling_df['has_current_affirmed_complication'].sum():,}"
)

display(
    sampling_df[
        [
            "note_id",
            "subject_id",
            "word_count",
            "affirmed_complication_mentions",
            "categories_with_affirmed_mentions",
            "has_current_affirmed_complication",
        ]
    ].head()
)

**Defining the sampling design**

In [ ]:
# 14. Gold-Standard Sampling Configuration

RANDOM_SEED = 42

N_RANDOM_NOTES = 50
N_ENRICHED_NOTES = 20
N_TOTAL_NOTES = N_RANDOM_NOTES + N_ENRICHED_NOTES

MINIMUM_WORD_COUNT = 250
MAXIMUM_WORD_COUNT = 5_000

sampling_configuration = {
    "random_seed": RANDOM_SEED,
    "random_notes": N_RANDOM_NOTES,
    "category_enriched_notes": N_ENRICHED_NOTES,
    "total_notes": N_TOTAL_NOTES,
    "minimum_word_count": MINIMUM_WORD_COUNT,
    "maximum_word_count": MAXIMUM_WORD_COUNT,
    "one_note_per_patient": True,
}

display(
    pd.DataFrame(
        sampling_configuration.items(),
        columns=["parameter", "value"],
    )
)

**Building the eligible cohort and random sample**

In [ ]:
# 15. Selecting the 50-Note Random Sample

eligible_df = sampling_df.loc[
    sampling_df["word_count"].between(
        MINIMUM_WORD_COUNT,
        MAXIMUM_WORD_COUNT,
        inclusive="both",
    )
].copy()

# Randomising before retaining one note per patient.
eligible_randomised_df = eligible_df.sample(
    frac=1,
    random_state=RANDOM_SEED,
)

unique_patient_pool_df = (
    eligible_randomised_df
    .drop_duplicates(
        subset=["subject_id"],
        keep="first",
    )
    .reset_index(drop=True)
)

random_sample_df = unique_patient_pool_df.sample(
    n=N_RANDOM_NOTES,
    random_state=RANDOM_SEED,
    replace=False,
).copy()

random_sample_df["sampling_group"] = "random"
random_sample_df["enrichment_category"] = pd.NA

print("-" * 90)
print("RANDOM SAMPLE")
print("-" * 90)

print(f"Eligible notes             : {len(eligible_df):,}")
print(f"Eligible unique patients   : {len(unique_patient_pool_df):,}")
print(f"Random notes selected      : {len(random_sample_df):,}")
print(f"Random unique patients     : {random_sample_df['subject_id'].nunique():,}")
print(
    "Random notes with affirmed complications: "
    f"{random_sample_df['has_current_affirmed_complication'].sum():,}"
)

display(
    random_sample_df[
        [
            "note_id",
            "subject_id",
            "word_count",
            "has_current_affirmed_complication",
            "affirmed_complication_mentions",
        ]
    ].sort_values("note_id").head(10)
)

**Selecting 20 category-enriched notes**

This selects one affirmed note from up to 20 complication categories, prioritising less frequent categories.

In [ ]:
# 16. Selecting the 20-Note Category-Enriched Sample

import numpy as np

affirmed_note_category_df = note_category_df.loc[
    note_category_df["has_affirmed_mention"].astype(bool)
].copy()

affirmed_note_category_df = affirmed_note_category_df.merge(
    category_prevalence_df[
        [
            "category",
            "notes_with_affirmed_mention",
            "note_prevalence_affirmed_percentage",
        ]
    ],
    on="category",
    how="left",
    validate="many_to_one",
)

# Restricted to notes in the eligible cohort.
affirmed_note_category_df = affirmed_note_category_df.loc[
    affirmed_note_category_df["note_id"].isin(
        set(eligible_df["note_id"])
    )
].copy()

# Excluding notes and patients already selected randomly.
random_note_ids = set(random_sample_df["note_id"])
random_patient_ids = set(random_sample_df["subject_id"])

affirmed_note_category_df = affirmed_note_category_df.loc[
    ~affirmed_note_category_df["note_id"].isin(random_note_ids)
].copy()

affirmed_note_category_df = affirmed_note_category_df.merge(
    eligible_df[
        [
            "note_id",
            "subject_id",
            "word_count",
            "categories_with_affirmed_mentions",
        ]
    ],
    on=["note_id", "subject_id"],
    how="inner",
)

affirmed_note_category_df = affirmed_note_category_df.loc[
    ~affirmed_note_category_df["subject_id"].isin(
        random_patient_ids
    )
].copy()

# Prioritising categories with lower affirmed prevalence.
category_order = (
    affirmed_note_category_df[
        [
            "category",
            "notes_with_affirmed_mention",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "notes_with_affirmed_mention",
            "category",
        ],
        ascending=[True, True],
    )
    ["category"]
    .tolist()
)

selected_records = []
selected_note_ids = set()
selected_patient_ids = set(random_patient_ids)

rng = np.random.default_rng(RANDOM_SEED)

for category in category_order:

    if len(selected_records) >= N_ENRICHED_NOTES:
        break

    candidates_df = affirmed_note_category_df.loc[
        (affirmed_note_category_df["category"] == category)
        & (~affirmed_note_category_df["note_id"].isin(selected_note_ids))
        & (~affirmed_note_category_df["subject_id"].isin(selected_patient_ids))
    ].copy()

    if candidates_df.empty:
        continue

    # Prefering notes with fewer concurrent categories, making annotation clearer.
    minimum_category_count = (
        candidates_df["categories_with_affirmed_mentions"].min()
    )

    preferred_candidates_df = candidates_df.loc[
        candidates_df["categories_with_affirmed_mentions"]
        == minimum_category_count
    ]

    chosen_index = rng.choice(
        preferred_candidates_df.index.to_numpy()
    )

    chosen_row = preferred_candidates_df.loc[chosen_index]

    selected_records.append(
        {
            "note_id": chosen_row["note_id"],
            "enrichment_category": category,
        }
    )

    selected_note_ids.add(chosen_row["note_id"])
    selected_patient_ids.add(chosen_row["subject_id"])

if len(selected_records) != N_ENRICHED_NOTES:
    raise ValueError(
        f"Requested {N_ENRICHED_NOTES} enriched notes, "
        f"but selected only {len(selected_records)}."
    )

enriched_selection_df = pd.DataFrame(selected_records)

enriched_sample_df = eligible_df.merge(
    enriched_selection_df,
    on="note_id",
    how="inner",
    validate="one_to_one",
)

enriched_sample_df["sampling_group"] = "category_enriched"

print("-" * 90)
print("CATEGORY-ENRICHED SAMPLE")
print("-" * 90)

print(f"Notes selected          : {len(enriched_sample_df):,}")
print(f"Unique patients         : {enriched_sample_df['subject_id'].nunique():,}")
print(
    f"Categories represented  : "
    f"{enriched_sample_df['enrichment_category'].nunique():,}"
)

display(
    enriched_sample_df[
        [
            "note_id",
            "subject_id",
            "word_count",
            "enrichment_category",
            "categories_with_affirmed_mentions",
        ]
    ]
    .sort_values("enrichment_category")
    .reset_index(drop=True)
)


**Combining and validating the final sample**

In [ ]:
# 17. Combining and Validating the 70-Note Annotation Sample

annotation_sample_df = pd.concat(
    [
        random_sample_df,
        enriched_sample_df,
    ],
    ignore_index=True,
)

annotation_sample_df = annotation_sample_df.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(drop=True)

annotation_sample_df.insert(
    0,
    "annotation_note_number",
    np.arange(1, len(annotation_sample_df) + 1),
)

final_validation = {
    "total_notes_equals_70":
        len(annotation_sample_df) == N_TOTAL_NOTES,

    "note_ids_are_unique":
        annotation_sample_df["note_id"].is_unique,

    "patients_are_unique":
        annotation_sample_df["subject_id"].is_unique,

    "random_notes_equal_50":
        annotation_sample_df["sampling_group"]
        .eq("random")
        .sum() == N_RANDOM_NOTES,

    "enriched_notes_equal_20":
        annotation_sample_df["sampling_group"]
        .eq("category_enriched")
        .sum() == N_ENRICHED_NOTES,

    "all_notes_have_text":
        annotation_sample_df["text"]
        .str.strip()
        .ne("")
        .all(),

    "all_notes_within_word_limits":
        annotation_sample_df["word_count"]
        .between(
            MINIMUM_WORD_COUNT,
            MAXIMUM_WORD_COUNT,
            inclusive="both",
        )
        .all(),
}

final_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check,
            "passed": bool(result),
        }
        for check, result in final_validation.items()
    ]
)

display(final_validation_df)

if not final_validation_df["passed"].all():
    raise ValueError(
        "Final annotation sample validation failed."
    )

print("\nFinal 70-note annotation sample validated successfully.")

print("\nSample composition:")

display(
    annotation_sample_df[
        "sampling_group"
    ]
    .value_counts()
    .rename_axis("sampling_group")
    .reset_index(name="note_count")
)

print(
    f"\nTotal annotation words: "
    f"{annotation_sample_df['word_count'].sum():,}"
)

print(
    f"Median words per note: "
    f"{annotation_sample_df['word_count'].median():,.0f}"
)

**Exporting and freezing the sample manifest**

In [ ]:
# 18. Freezing and Exporting the Annotation Sample

from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

SAMPLE_MANIFEST_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_70_note_sample_manifest.csv"
)

SAMPLE_TEXT_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_70_note_sample_with_text.csv.gz"
)

SAMPLE_CONFIG_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_sampling_configuration.json"
)

manifest_columns = [
    "annotation_note_number",
    "note_id",
    "subject_id",
    "hadm_id",
    "sampling_group",
    "enrichment_category",
    "word_count",
    "text_character_count",
    "total_complication_mentions",
    "affirmed_complication_mentions",
    "unique_complication_categories",
    "categories_with_affirmed_mentions",
    "has_any_complication_mention",
    "has_current_affirmed_complication",
]

sample_manifest_df = annotation_sample_df[
    manifest_columns
].copy()

sample_with_text_df = annotation_sample_df[
    manifest_columns + ["text"]
].copy()

sample_manifest_df.to_csv(
    SAMPLE_MANIFEST_FILE,
    index=False,
)

sample_with_text_df.to_csv(
    SAMPLE_TEXT_FILE,
    index=False,
    compression="gzip",
)

sample_hash = hashlib.sha256(
    sample_manifest_df
    .sort_values("annotation_note_number")
    .to_csv(index=False)
    .encode("utf-8")
).hexdigest()

sampling_metadata = {
    **sampling_configuration,
    "creation_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "sample_manifest_sha256": sample_hash,
    "primary_evaluation_subset": "random",
    "supplementary_evaluation_subset": "category_enriched",
    "random_subset_size": 50,
    "enriched_subset_size": 20,
    "total_annotation_words": int(
        annotation_sample_df["word_count"].sum()
    ),
    "median_words_per_note": float(
        annotation_sample_df["word_count"].median()
    ),
}

with open(
    SAMPLE_CONFIG_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        sampling_metadata,
        file,
        indent=2,
    )

print("-" * 90)
print("FROZEN SAMPLE OUTPUTS")
print("-" * 90)

print(f"Manifest       : {SAMPLE_MANIFEST_FILE}")
print(f"Text dataset   : {SAMPLE_TEXT_FILE}")
print(f"Configuration  : {SAMPLE_CONFIG_FILE}")
print(f"SHA-256        : {sample_hash}")

**Validation of the frozen files**

In [ ]:
# 19. Validating Frozen Sample Files

reloaded_manifest_df = pd.read_csv(
    SAMPLE_MANIFEST_FILE,
    dtype={
        "note_id": str,
        "subject_id": str,
        "hadm_id": str,
    },
)

reloaded_text_df = pd.read_csv(
    SAMPLE_TEXT_FILE,
    compression="gzip",
    dtype={
        "note_id": str,
        "subject_id": str,
        "hadm_id": str,
    },
)

freeze_validation = {
    "manifest_has_70_rows":
        len(reloaded_manifest_df) == 70,

    "text_file_has_70_rows":
        len(reloaded_text_df) == 70,

    "manifest_note_ids_unique":
        reloaded_manifest_df["note_id"].is_unique,

    "text_note_ids_unique":
        reloaded_text_df["note_id"].is_unique,

    "manifest_and_text_ids_match":
        set(reloaded_manifest_df["note_id"])
        == set(reloaded_text_df["note_id"]),

    "all_text_present":
        reloaded_text_df["text"]
        .fillna("")
        .str.strip()
        .ne("")
        .all(),
}

freeze_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check,
            "passed": bool(result),
        }
        for check, result in freeze_validation.items()
    ]
)

display(freeze_validation_df)

if not freeze_validation_df["passed"].all():
    raise ValueError(
        "Frozen sample validation failed."
    )

print("\nFrozen annotation sample validated successfully.")

**Defining the annotation labels**

In [ ]:
# 20. Defining Gold-Standard Annotation Labels

COMPLICATION_CATEGORIES = sorted(
    category_prevalence_df["category"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

ASSERTION_LABELS = [
    "AFFIRMED",
    "NEGATED",
    "UNCERTAIN",
    "HYPOTHETICAL",
    "HISTORICAL",
    "FAMILY",
]

TEMPORALITY_LABELS = [
    "CURRENT",
    "HISTORICAL",
    "PLANNED_OR_FUTURE",
    "UNCLEAR",
]

RELEVANCE_LABELS = [
    "CLINICAL_COMPLICATION",
    "NOT_A_COMPLICATION",
    "UNCLEAR",
]

ANNOTATION_STATUS_LABELS = [
    "NOT_STARTED",
    "IN_PROGRESS",
    "COMPLETED",
    "REVIEW_REQUIRED",
]

print(f"Complication categories : {len(COMPLICATION_CATEGORIES)}")
print(f"Assertion labels         : {len(ASSERTION_LABELS)}")
print(f"Temporality labels       : {len(TEMPORALITY_LABELS)}")
print(f"Relevance labels         : {len(RELEVANCE_LABELS)}")

display(
    pd.DataFrame(
        {
            "complication_category":
                COMPLICATION_CATEGORIES
        }
    )
)

**Creating the annotation workbook**

This workbook will contain:

* Instructions
* Note_Register
* Mention_Annotations
* Category_Definitions
* Validation_Lists




In [ ]:
# 21. Generating the Manual Annotation Workbook


import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side,
)
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

ANNOTATION_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_annotation_workbook.xlsx"
)

workbook = Workbook()

# Remove default sheet.
default_sheet = workbook.active
workbook.remove(default_sheet)

instructions_ws = workbook.create_sheet("Instructions")
register_ws = workbook.create_sheet("Note_Register")
annotations_ws = workbook.create_sheet("Mention_Annotations")
definitions_ws = workbook.create_sheet("Category_Definitions")
validation_ws = workbook.create_sheet("Validation_Lists")

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78",
)

subheader_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7",
)

editable_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC",
)

header_font = Font(
    color="FFFFFF",
    bold=True,
)

bold_font = Font(bold=True)

thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)


# Instructions sheet


instructions = [
    (
        "Gold-Standard Clinical Complication Annotation",
        ""
    ),
    (
        "Purpose",
        "Manually identify clinical complication mentions "
        "in ICU discharge summaries."
    ),
    (
        "Primary evaluation subset",
        "The 50 randomly sampled notes."
    ),
    (
        "Supplementary subset",
        "The 20 category-enriched notes."
    ),
    (
        "Annotation unit",
        "One explicit textual mention of a complication."
    ),
    (
        "Use original text",
        "Annotate only the original discharge-summary text "
        "shown in the Note_Register sheet."
    ),
    (
        "Category",
        "Choose the most appropriate complication category."
    ),
    (
        "Assertion",
        "Record whether the mention is affirmed, negated, "
        "uncertain, hypothetical, historical or family-related."
    ),
    (
        "Current affirmed complication",
        "Use AFFIRMED with CURRENT temporality when the note "
        "describes an actual complication relevant to the admission."
    ),
    (
        "Do not infer",
        "Do not annotate a complication unless it is explicitly "
        "supported by the note."
    ),
    (
        "Duplicate mentions",
        "Annotate each distinct mention occurrence separately."
    ),
    (
        "Unclear cases",
        "Mark REVIEW_REQUIRED and explain the issue in annotator_notes."
    ),
]

for row_index, (heading, description) in enumerate(
    instructions,
    start=1,
):
    instructions_ws.cell(
        row=row_index,
        column=1,
        value=heading,
    )

    instructions_ws.cell(
        row=row_index,
        column=2,
        value=description,
    )

instructions_ws["A1"].font = Font(
    bold=True,
    size=16,
    color="FFFFFF",
)

instructions_ws["A1"].fill = header_fill
instructions_ws["B1"].fill = header_fill

for row in range(2, len(instructions) + 1):
    instructions_ws.cell(row, 1).font = bold_font
    instructions_ws.cell(row, 1).fill = subheader_fill

instructions_ws.column_dimensions["A"].width = 31
instructions_ws.column_dimensions["B"].width = 105

for row in instructions_ws.iter_rows():
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )
        cell.border = thin_border


# Note register


register_columns = [
    "annotation_note_number",
    "note_id",
    "subject_id",
    "hadm_id",
    "sampling_group",
    "enrichment_category",
    "word_count",
    "annotation_status",
    "annotator_id",
    "annotation_date",
    "review_notes",
    "original_note_text",
]

register_ws.append(register_columns)

for cell in register_ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )

for _, row in annotation_sample_df.sort_values(
    "annotation_note_number"
).iterrows():

    register_ws.append(
        [
            int(row["annotation_note_number"]),
            row["note_id"],
            row["subject_id"],
            row["hadm_id"],
            row["sampling_group"],
            row["enrichment_category"],
            int(row["word_count"]),
            "NOT_STARTED",
            "",
            "",
            "",
            row["text"],
        ]
    )

for row_number in range(
    2,
    register_ws.max_row + 1,
):
    for column_number in [8, 9, 10, 11]:
        register_ws.cell(
            row=row_number,
            column=column_number,
        ).fill = editable_fill

    register_ws.cell(
        row=row_number,
        column=12,
    ).alignment = Alignment(
        vertical="top",
        wrap_text=True,
    )

register_widths = {
    "A": 12,
    "B": 23,
    "C": 14,
    "D": 14,
    "E": 20,
    "F": 34,
    "G": 12,
    "H": 18,
    "I": 16,
    "J": 16,
    "K": 35,
    "L": 110,
}

for column, width in register_widths.items():
    register_ws.column_dimensions[column].width = width

register_ws.freeze_panes = "A2"
register_ws.auto_filter.ref = register_ws.dimensions


# Mention annotations


annotation_columns = [
    "annotation_id",
    "annotation_note_number",
    "note_id",
    "mention_text",
    "complication_category",
    "relevance",
    "assertion",
    "temporality",
    "start_char",
    "end_char",
    "supporting_sentence",
    "annotator_confidence",
    "review_required",
    "annotator_notes",
]

annotations_ws.append(annotation_columns)

for cell in annotations_ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )

# Pre-created 700 editable rows: approximately 10 mentions per note.
for row_number in range(2, 702):
    annotations_ws.cell(
        row=row_number,
        column=1,
        value=f'=IF(B{row_number}="","",'
              f'"ANN-"&TEXT(ROW()-1,"0000"))',
    )

    for column_number in range(2, 15):
        annotations_ws.cell(
            row=row_number,
            column=column_number,
        ).fill = editable_fill

annotation_widths = {
    "A": 15,
    "B": 13,
    "C": 23,
    "D": 30,
    "E": 34,
    "F": 24,
    "G": 18,
    "H": 20,
    "I": 12,
    "J": 12,
    "K": 70,
    "L": 20,
    "M": 18,
    "N": 50,
}

for column, width in annotation_widths.items():
    annotations_ws.column_dimensions[column].width = width

annotations_ws.freeze_panes = "A2"
annotations_ws.auto_filter.ref = f"A1:N701"

for row in annotations_ws.iter_rows(
    min_row=2,
    max_row=701,
    min_col=1,
    max_col=14,
):
    for cell in row:
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )
        cell.border = thin_border


# Validation lists


validation_lists = {
    "A": [
        "annotation_status",
        *ANNOTATION_STATUS_LABELS,
    ],
    "B": [
        "complication_category",
        *COMPLICATION_CATEGORIES,
    ],
    "C": [
        "relevance",
        *RELEVANCE_LABELS,
    ],
    "D": [
        "assertion",
        *ASSERTION_LABELS,
    ],
    "E": [
        "temporality",
        *TEMPORALITY_LABELS,
    ],
    "F": [
        "confidence",
        "HIGH",
        "MEDIUM",
        "LOW",
    ],
    "G": [
        "review_required",
        "YES",
        "NO",
    ],
}

for column, values in validation_lists.items():
    for row_number, value in enumerate(
        values,
        start=1,
    ):
        validation_ws[
            f"{column}{row_number}"
        ] = value

validation_ws.sheet_state = "hidden"


# Data validations

status_validation = DataValidation(
    type="list",
    formula1=(
        "'Validation_Lists'!"
        "$A$2:$A$5"
    ),
    allow_blank=False,
)

register_ws.add_data_validation(status_validation)
status_validation.add(
    f"H2:H{register_ws.max_row}"
)

category_validation = DataValidation(
    type="list",
    formula1=(
        f"'Validation_Lists'!"
        f"$B$2:$B${len(COMPLICATION_CATEGORIES) + 1}"
    ),
    allow_blank=True,
)

relevance_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$C$2:$C$4",
    allow_blank=True,
)

assertion_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$D$2:$D$7",
    allow_blank=True,
)

temporality_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$E$2:$E$5",
    allow_blank=True,
)

confidence_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$F$2:$F$4",
    allow_blank=True,
)

review_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$G$2:$G$3",
    allow_blank=True,
)

for validation in [
    category_validation,
    relevance_validation,
    assertion_validation,
    temporality_validation,
    confidence_validation,
    review_validation,
]:
    annotations_ws.add_data_validation(validation)

category_validation.add("E2:E701")
relevance_validation.add("F2:F701")
assertion_validation.add("G2:G701")
temporality_validation.add("H2:H701")
confidence_validation.add("L2:L701")
review_validation.add("M2:M701")


# Category definitions

definitions_ws.append(
    [
        "complication_category",
        "operational_definition",
        "include",
        "exclude_or_caution",
    ]
)

for cell in definitions_ws[1]:
    cell.fill = header_fill
    cell.font = header_font

for category in COMPLICATION_CATEGORIES:
    definitions_ws.append(
        [
            category,
            "",
            "",
            "",
        ]
    )

for row_number in range(
    2,
    definitions_ws.max_row + 1,
):
    for column_number in range(2, 5):
        definitions_ws.cell(
            row=row_number,
            column=column_number,
        ).fill = editable_fill

definitions_ws.column_dimensions["A"].width = 36
definitions_ws.column_dimensions["B"].width = 65
definitions_ws.column_dimensions["C"].width = 55
definitions_ws.column_dimensions["D"].width = 55

for worksheet in [
    register_ws,
    annotations_ws,
    definitions_ws,
]:
    for row in worksheet.iter_rows():
        for cell in row:
            cell.border = thin_border
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
            )

workbook.save(ANNOTATION_WORKBOOK_FILE)

print("-" * 90)
print("ANNOTATION WORKBOOK CREATED")
print("-" * 90)
print(ANNOTATION_WORKBOOK_FILE)

**Extracting medSpaCy candidates for the 70 notes**

In [ ]:
# 22. Extracting medSpaCy Candidate Mentions for the 70-Note Sample

import pandas as pd
import numpy as np
from pathlib import Path

MEDSPACY_SHARD_DIRECTORY = (
    MEDSPACY_DIRECTORY
    / "entity_shards"
)

CANDIDATE_MENTIONS_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_70_note_medspacy_candidates.csv.gz"
)

if not MEDSPACY_SHARD_DIRECTORY.is_dir():
    raise FileNotFoundError(
        "medSpaCy entity shard directory was not found:\n"
        f"{MEDSPACY_SHARD_DIRECTORY}"
    )

mention_shards = sorted(
    MEDSPACY_SHARD_DIRECTORY.glob(
        "medspacy_mentions_shard_*.csv.gz"
    )
)

if not mention_shards:
    raise FileNotFoundError(
        "No production medSpaCy mention shards were found."
    )

sample_note_ids = set(
    annotation_sample_df["note_id"].astype(str)
)

required_candidate_columns = [
    "note_id",
    "subject_id",
    "hadm_id",
    "entity_text",
    "normalized_text",
    "category",
    "start_char",
    "end_char",
    "sentence_text",
    "window_number",
    "is_negated",
    "is_historical",
    "is_family",
    "is_uncertain",
    "is_hypothetical",
    "is_current_affirmed",
]

candidate_frames = []
processed_shards = 0

for shard_path in mention_shards:

    shard_df = pd.read_csv(
        shard_path,
        usecols=required_candidate_columns,
        low_memory=False,
    )

    shard_df["note_id"] = (
        shard_df["note_id"]
        .astype(str)
    )

    selected_df = shard_df.loc[
        shard_df["note_id"].isin(sample_note_ids)
    ].copy()

    if not selected_df.empty:
        selected_df["source_shard"] = shard_path.name
        candidate_frames.append(selected_df)

    processed_shards += 1

if candidate_frames:
    candidate_mentions_df = pd.concat(
        candidate_frames,
        ignore_index=True,
    )
else:
    candidate_mentions_df = pd.DataFrame(
        columns=required_candidate_columns
        + ["source_shard"]
    )

# Standardising identifiers.
for column in [
    "note_id",
    "subject_id",
    "hadm_id",
]:
    candidate_mentions_df[column] = (
        candidate_mentions_df[column]
        .astype(str)
    )

# Removing exact duplicated candidates, if any.
candidate_mentions_df = (
    candidate_mentions_df
    .drop_duplicates(
        subset=[
            "note_id",
            "start_char",
            "end_char",
            "category",
            "entity_text",
        ]
    )
    .sort_values(
        [
            "note_id",
            "start_char",
            "end_char",
            "category",
        ]
    )
    .reset_index(drop=True)
)

candidate_mentions_df.insert(
    0,
    "candidate_id",
    [
        f"CAND-{number:05d}"
        for number in range(
            1,
            len(candidate_mentions_df) + 1,
        )
    ],
)

# Adding annotation-note number.
candidate_mentions_df = (
    candidate_mentions_df
    .merge(
        annotation_sample_df[
            [
                "annotation_note_number",
                "note_id",
                "sampling_group",
                "enrichment_category",
            ]
        ],
        on="note_id",
        how="left",
        validate="many_to_one",
    )
)

# Deriving the medSpaCy predicted assertion label.
def derive_predicted_assertion(row):

    if bool(row["is_family"]):
        return "FAMILY"

    if bool(row["is_hypothetical"]):
        return "HYPOTHETICAL"

    if bool(row["is_uncertain"]):
        return "UNCERTAIN"

    if bool(row["is_historical"]):
        return "HISTORICAL"

    if bool(row["is_negated"]):
        return "NEGATED"

    if bool(row["is_current_affirmed"]):
        return "AFFIRMED"

    return "UNCLEAR"


candidate_mentions_df["predicted_assertion"] = (
    candidate_mentions_df.apply(
        derive_predicted_assertion,
        axis=1,
    )
)

candidate_mentions_df["predicted_temporality"] = np.select(
    [
        candidate_mentions_df[
            "is_historical"
        ].astype(bool),

        candidate_mentions_df[
            "is_hypothetical"
        ].astype(bool),

        candidate_mentions_df[
            "is_current_affirmed"
        ].astype(bool),
    ],
    [
        "HISTORICAL",
        "PLANNED_OR_FUTURE",
        "CURRENT",
    ],
    default="UNCLEAR",
)

candidate_mentions_df.to_csv(
    CANDIDATE_MENTIONS_FILE,
    index=False,
    compression="gzip",
)

print("-" * 90)
print("MEDSPACY CANDIDATE EXTRACTION")
print("-" * 90)

print(f"Production shards processed : {processed_shards:,}")
print(f"Sample notes                : {len(sample_note_ids):,}")
print(f"Candidate mentions          : {len(candidate_mentions_df):,}")
print(
    "Notes containing candidates : "
    f"{candidate_mentions_df['note_id'].nunique():,}"
)
print(
    "Notes without candidates     : "
    f"{len(sample_note_ids - set(candidate_mentions_df['note_id'])):,}"
)
print(
    "Categories represented       : "
    f"{candidate_mentions_df['category'].nunique():,}"
)
print(f"Output file                 : {CANDIDATE_MENTIONS_FILE}")

display(
    candidate_mentions_df[
        [
            "candidate_id",
            "annotation_note_number",
            "note_id",
            "entity_text",
            "category",
            "predicted_assertion",
            "predicted_temporality",
            "sentence_text",
        ]
    ].head(10)
)

**Creating the candidate-assisted review workbook**

This preserves the blank Mention Annotations sheet for manually identified missed mentions and adds a separate Candidate Review sheet.

In [ ]:
# 23. Creating Candidate-Assisted Gold-Standard Review Workbook


from openpyxl import load_workbook
from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side,
)
from openpyxl.worksheet.datavalidation import DataValidation

PREANNOTATED_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook.xlsx"
)

# Loading the blank workbook produced earlier.
workbook = load_workbook(
    ANNOTATION_WORKBOOK_FILE
)

# Removing old versions if this cell is rerun.
for sheet_name in [
    "Candidate_Review",
    "Full_Text_Review",
]:
    if sheet_name in workbook.sheetnames:
        del workbook[sheet_name]

candidate_ws = workbook.create_sheet(
    "Candidate_Review",
    2,
)

full_review_ws = workbook.create_sheet(
    "Full_Text_Review",
    3,
)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78",
)

header_font = Font(
    color="FFFFFF",
    bold=True,
)

editable_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC",
)

prediction_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7",
)

thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)


# Candidate review sheet


candidate_review_columns = [
    "candidate_id",
    "annotation_note_number",
    "note_id",
    "sampling_group",
    "enrichment_category",
    "mention_text",
    "predicted_category",
    "predicted_assertion",
    "predicted_temporality",
    "start_char",
    "end_char",
    "supporting_sentence",
    "reviewer_decision",
    "gold_mention_text",
    "gold_category",
    "gold_relevance",
    "gold_assertion",
    "gold_temporality",
    "annotator_confidence",
    "review_required",
    "annotator_notes",
]

candidate_ws.append(candidate_review_columns)

for cell in candidate_ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )

for _, row in candidate_mentions_df.iterrows():

    candidate_ws.append(
        [
            row["candidate_id"],
            int(row["annotation_note_number"]),
            row["note_id"],
            row["sampling_group"],
            row["enrichment_category"],
            row["entity_text"],
            row["category"],
            row["predicted_assertion"],
            row["predicted_temporality"],
            int(row["start_char"]),
            int(row["end_char"]),
            row["sentence_text"],
            "",
            row["entity_text"],
            row["category"],
            "CLINICAL_COMPLICATION",
            row["predicted_assertion"],
            row["predicted_temporality"],
            "",
            "NO",
            "",
        ]
    )

# Prediction columns: blue.
prediction_columns = list(range(1, 13))

# Reviewer-editable columns: yellow.
editable_columns = list(range(13, 22))

for row_number in range(
    2,
    candidate_ws.max_row + 1,
):
    for column_number in prediction_columns:
        candidate_ws.cell(
            row=row_number,
            column=column_number,
        ).fill = prediction_fill

    for column_number in editable_columns:
        candidate_ws.cell(
            row=row_number,
            column=column_number,
        ).fill = editable_fill

candidate_widths = {
    "A": 16,
    "B": 13,
    "C": 23,
    "D": 20,
    "E": 32,
    "F": 28,
    "G": 34,
    "H": 22,
    "I": 22,
    "J": 11,
    "K": 11,
    "L": 70,
    "M": 18,
    "N": 28,
    "O": 34,
    "P": 24,
    "Q": 18,
    "R": 22,
    "S": 20,
    "T": 18,
    "U": 50,
}

for column, width in candidate_widths.items():
    candidate_ws.column_dimensions[column].width = width

candidate_ws.freeze_panes = "A2"
candidate_ws.auto_filter.ref = candidate_ws.dimensions

for row in candidate_ws.iter_rows():
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )


# Full-text review sheet


full_review_columns = [
    "annotation_note_number",
    "note_id",
    "sampling_group",
    "enrichment_category",
    "candidate_count",
    "candidate_review_complete",
    "full_original_text_reviewed",
    "missed_mentions_added",
    "note_annotation_complete",
    "annotator_id",
    "review_date",
    "review_notes",
]

full_review_ws.append(full_review_columns)

candidate_counts = (
    candidate_mentions_df
    .groupby("note_id")
    .size()
    .rename("candidate_count")
)

for _, row in (
    annotation_sample_df
    .sort_values("annotation_note_number")
    .iterrows()
):
    full_review_ws.append(
        [
            int(row["annotation_note_number"]),
            row["note_id"],
            row["sampling_group"],
            row["enrichment_category"],
            int(candidate_counts.get(row["note_id"], 0)),
            "NO",
            "NO",
            "NO",
            "NO",
            "",
            "",
            "",
        ]
    )

for row_number in range(
    2,
    full_review_ws.max_row + 1,
):
    for column_number in range(6, 13):
        full_review_ws.cell(
            row=row_number,
            column=column_number,
        ).fill = editable_fill

full_review_widths = {
    "A": 13,
    "B": 23,
    "C": 20,
    "D": 34,
    "E": 16,
    "F": 24,
    "G": 25,
    "H": 22,
    "I": 24,
    "J": 16,
    "K": 16,
    "L": 50,
}

for column, width in full_review_widths.items():
    full_review_ws.column_dimensions[column].width = width

full_review_ws.freeze_panes = "A2"
full_review_ws.auto_filter.ref = full_review_ws.dimensions

for row in full_review_ws.iter_rows():
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )


# Adding validation lists


validation_ws = workbook["Validation_Lists"]

additional_validation_lists = {
    "H": [
        "reviewer_decision",
        "KEEP",
        "MODIFY",
        "DELETE",
        "REVIEW_REQUIRED",
    ],
    "I": [
        "yes_no",
        "YES",
        "NO",
    ],
}

for column, values in additional_validation_lists.items():
    for row_number, value in enumerate(
        values,
        start=1,
    ):
        validation_ws[
            f"{column}{row_number}"
        ] = value

decision_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$H$2:$H$5",
    allow_blank=True,
)

category_validation = DataValidation(
    type="list",
    formula1=(
        "'Validation_Lists'!"
        f"$B$2:$B${len(COMPLICATION_CATEGORIES) + 1}"
    ),
    allow_blank=True,
)

relevance_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$C$2:$C$4",
    allow_blank=True,
)

assertion_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$D$2:$D$7",
    allow_blank=True,
)

temporality_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$E$2:$E$5",
    allow_blank=True,
)

confidence_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$F$2:$F$4",
    allow_blank=True,
)

review_required_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$G$2:$G$3",
    allow_blank=True,
)

yes_no_validation = DataValidation(
    type="list",
    formula1="'Validation_Lists'!$I$2:$I$3",
    allow_blank=False,
)

for validation in [
    decision_validation,
    category_validation,
    relevance_validation,
    assertion_validation,
    temporality_validation,
    confidence_validation,
    review_required_validation,
]:
    candidate_ws.add_data_validation(validation)

decision_validation.add(
    f"M2:M{candidate_ws.max_row}"
)

category_validation.add(
    f"O2:O{candidate_ws.max_row}"
)

relevance_validation.add(
    f"P2:P{candidate_ws.max_row}"
)

assertion_validation.add(
    f"Q2:Q{candidate_ws.max_row}"
)

temporality_validation.add(
    f"R2:R{candidate_ws.max_row}"
)

confidence_validation.add(
    f"S2:S{candidate_ws.max_row}"
)

review_required_validation.add(
    f"T2:T{candidate_ws.max_row}"
)

full_review_ws.add_data_validation(
    yes_no_validation
)

for column in ["F", "G", "H", "I"]:
    yes_no_validation.add(
        f"{column}2:{column}{full_review_ws.max_row}"
    )


# Update instructions


instructions_ws = workbook["Instructions"]

additional_instructions = [
    (
        "Candidate review",
        "Review every row in Candidate_Review and choose KEEP, "
        "MODIFY, DELETE or REVIEW_REQUIRED."
    ),
    (
        "False-negative search",
        "After candidate review, read the entire original note and "
        "add any missed complication mentions to Mention_Annotations."
    ),
    (
        "Evaluation safeguard",
        "A note is complete only after both candidate review and "
        "independent full-text review have been completed."
    ),
    (
        "KEEP",
        "The candidate span, category and contextual labels are correct."
    ),
    (
        "MODIFY",
        "The candidate represents a valid mention, but one or more gold "
        "fields must be corrected."
    ),
    (
        "DELETE",
        "The candidate is not a valid clinical complication mention."
    ),
]

start_row = instructions_ws.max_row + 2

for offset, (heading, description) in enumerate(
    additional_instructions,
):
    row_number = start_row + offset

    instructions_ws.cell(
        row=row_number,
        column=1,
        value=heading,
    )

    instructions_ws.cell(
        row=row_number,
        column=2,
        value=description,
    )

    instructions_ws.cell(
        row=row_number,
        column=1,
    ).font = Font(bold=True)

    instructions_ws.cell(
        row=row_number,
        column=1,
    ).fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )

    for column_number in [1, 2]:
        instructions_ws.cell(
            row=row_number,
            column=column_number,
        ).border = thin_border

        instructions_ws.cell(
            row=row_number,
            column=column_number,
        ).alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )

workbook.save(
    PREANNOTATED_WORKBOOK_FILE
)

print("-" * 90)
print("CANDIDATE-ASSISTED WORKBOOK CREATED")
print("-" * 90)

print(f"Workbook            : {PREANNOTATED_WORKBOOK_FILE}")
print(f"Candidate rows       : {len(candidate_mentions_df):,}")
print(f"Full-text review rows: {len(annotation_sample_df):,}")
print(
    "Sheets              : "
    + ", ".join(workbook.sheetnames)
)

**Validation and document the annotation package**

In [ ]:
# 24. Validation and Documentation of the Notebook 06 Annotation Package


import json
import hashlib
from datetime import datetime, timezone
from openpyxl import load_workbook

ANNOTATION_PACKAGE_REPORT_FILE = (
    ANNOTATION_DIRECTORY
    / "notebook_06_annotation_package_report.json"
)

ANNOTATION_README_FILE = (
    ANNOTATION_DIRECTORY
    / "README_annotation_workflow.txt"
)

# Reloading workbook to verify that it is readable.
validation_workbook = load_workbook(
    PREANNOTATED_WORKBOOK_FILE,
    read_only=True,
    data_only=False,
)

required_sheets = {
    "Instructions",
    "Note_Register",
    "Candidate_Review",
    "Full_Text_Review",
    "Mention_Annotations",
    "Category_Definitions",
    "Validation_Lists",
}

actual_sheets = set(
    validation_workbook.sheetnames
)

candidate_ws_check = validation_workbook[
    "Candidate_Review"
]

full_review_ws_check = validation_workbook[
    "Full_Text_Review"
]

package_validation = {
    "candidate_file_exists":
        CANDIDATE_MENTIONS_FILE.is_file(),

    "candidate_workbook_exists":
        PREANNOTATED_WORKBOOK_FILE.is_file(),

    "required_sheets_present":
        required_sheets.issubset(actual_sheets),

    "candidate_sheet_row_count_correct":
        candidate_ws_check.max_row
        == len(candidate_mentions_df) + 1,

    "full_review_sheet_has_70_notes":
        full_review_ws_check.max_row == 71,

    "sample_manifest_exists":
        SAMPLE_MANIFEST_FILE.is_file(),

    "sample_text_file_exists":
        SAMPLE_TEXT_FILE.is_file(),

    "sampling_configuration_exists":
        SAMPLE_CONFIG_FILE.is_file(),
}

package_validation_df = pd.DataFrame(
    [
        {
            "validation_check": check,
            "passed": bool(result),
        }
        for check, result in package_validation.items()
    ]
)

display(package_validation_df)

if not package_validation_df["passed"].all():
    failed = package_validation_df.loc[
        ~package_validation_df["passed"],
        "validation_check",
    ].tolist()

    raise ValueError(
        "Annotation package validation failed:\n- "
        + "\n- ".join(failed)
    )

def file_sha256(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            hasher.update(block)

    return hasher.hexdigest()


package_report = {
    "notebook": "Notebook 06 - Gold-Standard Annotation",
    "technical_generation_status": "COMPLETED",
    "manual_annotation_status": "NOT_STARTED",
    "generated_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "sample_notes": int(len(annotation_sample_df)),
    "random_notes": int(
        annotation_sample_df[
            "sampling_group"
        ].eq("random").sum()
    ),
    "category_enriched_notes": int(
        annotation_sample_df[
            "sampling_group"
        ].eq("category_enriched").sum()
    ),
    "candidate_mentions": int(
        len(candidate_mentions_df)
    ),
    "complication_categories": int(
        len(COMPLICATION_CATEGORIES)
    ),
    "total_annotation_words": int(
        annotation_sample_df["word_count"].sum()
    ),
    "primary_evaluation_subset": "50 random notes",
    "supplementary_evaluation_subset":
        "20 category-enriched notes",
    "required_manual_steps": [
        "Complete Candidate_Review for every candidate.",
        "Read every original discharge summary in full.",
        "Add medSpaCy false negatives to Mention_Annotations.",
        "Complete Full_Text_Review for all 70 notes.",
        "Resolve all REVIEW_REQUIRED cases.",
        "Populate and freeze operational category definitions.",
    ],
    "files": {
        "sample_manifest": str(
            SAMPLE_MANIFEST_FILE
        ),
        "sample_text": str(
            SAMPLE_TEXT_FILE
        ),
        "sampling_configuration": str(
            SAMPLE_CONFIG_FILE
        ),
        "candidate_mentions": str(
            CANDIDATE_MENTIONS_FILE
        ),
        "blank_annotation_workbook": str(
            ANNOTATION_WORKBOOK_FILE
        ),
        "candidate_assisted_workbook": str(
            PREANNOTATED_WORKBOOK_FILE
        ),
    },
    "sha256": {
        "sample_manifest": file_sha256(
            SAMPLE_MANIFEST_FILE
        ),
        "candidate_mentions": file_sha256(
            CANDIDATE_MENTIONS_FILE
        ),
        "candidate_assisted_workbook": file_sha256(
            PREANNOTATED_WORKBOOK_FILE
        ),
    },
}

with open(
    ANNOTATION_PACKAGE_REPORT_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        package_report,
        file,
        indent=2,
    )

readme_text = """
NOTEBOOK 06 — GOLD-STANDARD ANNOTATION WORKFLOW

STATUS
Technical package generation: COMPLETE
Manual gold-standard annotation: NOT STARTED

FILES
1. gold_standard_70_note_sample_manifest.csv
   Frozen manifest for the 70 sampled discharge summaries.

2. gold_standard_70_note_sample_with_text.csv.gz
   Original discharge-summary text for all sampled notes.

3. gold_standard_70_note_medspacy_candidates.csv.gz
   Candidate complication mentions extracted from all production medSpaCy shards.

4. gold_standard_candidate_assisted_workbook.xlsx
   Main workbook for human verification and manual false-negative annotation.

ANNOTATION PROCEDURE
1. Open Note_Register and read the original discharge-summary text.
2. Review every candidate in Candidate_Review.
3. Select KEEP, MODIFY, DELETE or REVIEW_REQUIRED.
4. Correct the gold fields when MODIFY is selected.
5. Read the complete note independently of the candidate list.
6. Enter any missed complication mentions in Mention_Annotations.
7. Complete the Full_Text_Review checklist.
8. Resolve every REVIEW_REQUIRED annotation.
9. Mark the note complete only after all review stages are finished.

EVALUATION DESIGN
The 50-note random subset is the primary evaluation dataset.
The 20-note enriched subset is a supplementary category-coverage dataset.
Combined results must not be interpreted as an unbiased prevalence estimate.

IMPORTANT
The gold standard is not complete until every note has undergone full-text
review. Candidate verification alone is insufficient because it cannot detect
all false-negative medSpaCy mentions.
""".strip()

with open(
    ANNOTATION_README_FILE,
    "w",
    encoding="utf-8",
) as file:
    file.write(readme_text)

print("-" * 90)
print("NOTEBOOK 06 ANNOTATION PACKAGE VALIDATED")
print("-" * 90)

print(
    "Technical generation status : COMPLETED"
)
print(
    "Manual annotation status    : NOT STARTED"
)
print(
    f"Package report              : "
    f"{ANNOTATION_PACKAGE_REPORT_FILE}"
)
print(
    f"Workflow README             : "
    f"{ANNOTATION_README_FILE}"
)
print(
    f"Candidate workbook          : "
    f"{PREANNOTATED_WORKBOOK_FILE}"
)

**Audit and refining the generated workbook**

This cell:

* loads the workbook produced by Notebook 06;
* detects Excel formula-like clinical text;
* converts unsafe formulas back to literal text;
* checks the expected sheets;
* confirms 70 notes and 380 candidate rows;
* verifies the 50/20 subset split;
* adds a reproducible Audit_Summary sheet;
* creates the official refined workbook.

In [ ]:
# 25. Audit and Refinement of the Candidate-Assisted Workbook


from pathlib import Path
from datetime import datetime, timezone
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side

SOURCE_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook.xlsx"
)

FINAL_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook_final.xlsx"
)

if not SOURCE_WORKBOOK_FILE.is_file():
    raise FileNotFoundError(
        "The original candidate-assisted workbook was not found:\n"
        f"{SOURCE_WORKBOOK_FILE}"
    )

workbook = load_workbook(
    SOURCE_WORKBOOK_FILE,
    data_only=False,
)

required_sheets = [
    "Instructions",
    "Note_Register",
    "Candidate_Review",
    "Full_Text_Review",
    "Mention_Annotations",
    "Category_Definitions",
    "Validation_Lists",
]

missing_sheets = [
    sheet_name
    for sheet_name in required_sheets
    if sheet_name not in workbook.sheetnames
]

if missing_sheets:
    raise ValueError(
        "Required workbook sheets are missing:\n- "
        + "\n- ".join(missing_sheets)
    )


# Repairing formula-like clinical text


text_columns_to_check = {
    "Note_Register": ["L"],
    "Candidate_Review": ["F", "L", "N", "U"],
    "Mention_Annotations": ["D", "K", "N"],
    "Instructions": ["B"],
    "Category_Definitions": ["B", "C", "D"],
}

repaired_cells = []

for sheet_name, columns in text_columns_to_check.items():

    worksheet = workbook[sheet_name]

    for column_letter in columns:

        for row_number in range(
            2,
            worksheet.max_row + 1,
        ):
            cell = worksheet[
                f"{column_letter}{row_number}"
            ]

            value = cell.value

            # A text value starting with "=" may have been loaded by Excel
            # as a formula. Convert it to explicit literal text.
            if (
                isinstance(value, str)
                and cell.data_type == "f"
            ):
                literal_text = "=" + value.lstrip("=")

                cell.value = literal_text
                cell.data_type = "s"

                repaired_cells.append(
                    {
                        "sheet": sheet_name,
                        "cell": cell.coordinate,
                        "repair": "formula_to_literal_text",
                    }
                )


# Workbook content checks


note_register_ws = workbook["Note_Register"]
candidate_ws = workbook["Candidate_Review"]
full_review_ws = workbook["Full_Text_Review"]
mention_ws = workbook["Mention_Annotations"]
definitions_ws = workbook["Category_Definitions"]

note_register_rows = note_register_ws.max_row - 1
candidate_rows = candidate_ws.max_row - 1
full_review_rows = full_review_ws.max_row - 1
category_rows = definitions_ws.max_row - 1

# Read sample composition directly from Note_Register.
sampling_groups = []

for row_number in range(
    2,
    note_register_ws.max_row + 1,
):
    sampling_group = note_register_ws[
        f"E{row_number}"
    ].value

    if sampling_group is not None:
        sampling_groups.append(
            str(sampling_group).strip()
        )

random_note_count = sampling_groups.count("random")
enriched_note_count = sampling_groups.count(
    "category_enriched"
)

# Candidate decision status.
candidate_decisions = []

for row_number in range(
    2,
    candidate_ws.max_row + 1,
):
    decision = candidate_ws[
        f"M{row_number}"
    ].value

    if decision is not None and str(decision).strip():
        candidate_decisions.append(
            str(decision).strip()
        )

# Full-text review completion status.
completed_note_reviews = 0

for row_number in range(
    2,
    full_review_ws.max_row + 1,
):
    completion_value = full_review_ws[
        f"I{row_number}"
    ].value

    if str(completion_value).strip().upper() == "YES":
        completed_note_reviews += 1

# Category-definition completion status.
completed_category_definitions = 0

for row_number in range(
    2,
    definitions_ws.max_row + 1,
):
    definition = definitions_ws[
        f"B{row_number}"
    ].value
    include = definitions_ws[
        f"C{row_number}"
    ].value
    exclude = definitions_ws[
        f"D{row_number}"
    ].value

    if all(
        value is not None
        and str(value).strip()
        for value in [
            definition,
            include,
            exclude,
        ]
    ):
        completed_category_definitions += 1

audit_checks = {
    "required_sheets_present":
        not missing_sheets,

    "note_register_has_70_notes":
        note_register_rows == 70,

    "candidate_review_has_380_candidates":
        candidate_rows == 380,

    "full_text_review_has_70_notes":
        full_review_rows == 70,

    "category_definitions_has_23_categories":
        category_rows == 23,

    "random_subset_has_50_notes":
        random_note_count == 50,

    "enriched_subset_has_20_notes":
        enriched_note_count == 20,
}

failed_checks = [
    check
    for check, passed in audit_checks.items()
    if not passed
]

if failed_checks:
    raise ValueError(
        "Workbook audit failed:\n- "
        + "\n- ".join(failed_checks)
    )


# Creating Audit_Summary sheet


if "Audit_Summary" in workbook.sheetnames:
    del workbook["Audit_Summary"]

audit_ws = workbook.create_sheet(
    "Audit_Summary",
    1,
)

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78",
)

section_fill = PatternFill(
    fill_type="solid",
    fgColor="D9EAF7",
)

warning_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC",
)

header_font = Font(
    bold=True,
    color="FFFFFF",
)

bold_font = Font(
    bold=True,
)

thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

audit_rows = [
    [
        "Notebook 06 Workbook Audit",
        "",
    ],
    [
        "Audit timestamp UTC",
        datetime.now(timezone.utc).isoformat(),
    ],
    [
        "Source workbook",
        str(SOURCE_WORKBOOK_FILE),
    ],
    [
        "Official refined workbook",
        str(FINAL_WORKBOOK_FILE),
    ],
    [
        "",
        "",
    ],
    [
        "Technical audit result",
        "PASSED",
    ],
    [
        "Notes in sample",
        note_register_rows,
    ],
    [
        "Primary random notes",
        random_note_count,
    ],
    [
        "Supplementary enriched notes",
        enriched_note_count,
    ],
    [
        "Candidate mentions",
        candidate_rows,
    ],
    [
        "Complication categories",
        category_rows,
    ],
    [
        "Excel-unsafe cells repaired",
        len(repaired_cells),
    ],
    [
        "",
        "",
    ],
    [
        "Human annotation status",
        "NOT STARTED"
        if len(candidate_decisions) == 0
        else "IN PROGRESS",
    ],
    [
        "Candidate decisions completed",
        len(candidate_decisions),
    ],
    [
        "Candidate decisions remaining",
        candidate_rows - len(candidate_decisions),
    ],
    [
        "Full-text reviews completed",
        completed_note_reviews,
    ],
    [
        "Full-text reviews remaining",
        full_review_rows - completed_note_reviews,
    ],
    [
        "Complete category definitions",
        completed_category_definitions,
    ],
    [
        "Category definitions remaining",
        category_rows
        - completed_category_definitions,
    ],
    [
        "",
        "",
    ],
    [
        "Primary evaluation policy",
        "Use the 50 random notes for primary performance metrics.",
    ],
    [
        "Supplementary evaluation policy",
        "Report the 20 category-enriched notes separately as a "
        "challenge/category-coverage subset.",
    ],
    [
        "Notebook 07 readiness",
        "NOT READY — human annotation and category definitions "
        "must be completed first.",
    ],
]

for row in audit_rows:
    audit_ws.append(row)

audit_ws["A1"].fill = header_fill
audit_ws["B1"].fill = header_fill
audit_ws["A1"].font = Font(
    bold=True,
    color="FFFFFF",
    size=15,
)
audit_ws["B1"].font = header_font

for row_number in [
    6,
    14,
    22,
]:
    audit_ws[
        f"A{row_number}"
    ].fill = section_fill
    audit_ws[
        f"A{row_number}"
    ].font = bold_font

audit_ws["A24"].fill = warning_fill
audit_ws["B24"].fill = warning_fill
audit_ws["A24"].font = bold_font
audit_ws["B24"].font = bold_font

audit_ws.column_dimensions["A"].width = 38
audit_ws.column_dimensions["B"].width = 100

for row in audit_ws.iter_rows():
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )

audit_ws.freeze_panes = "A2"

# Save the official refined workbook.
workbook.save(
    FINAL_WORKBOOK_FILE
)

print("-" * 90)
print("WORKBOOK AUDIT AND REFINEMENT COMPLETED")
print("-" * 90)

print(f"Source workbook       : {SOURCE_WORKBOOK_FILE}")
print(f"Official workbook     : {FINAL_WORKBOOK_FILE}")
print(f"Technical checks      : {len(audit_checks)} passed")
print(f"Unsafe cells repaired : {len(repaired_cells)}")
print(f"Notes                 : {note_register_rows}")
print(f"Random notes          : {random_note_count}")
print(f"Enriched notes        : {enriched_note_count}")
print(f"Candidate mentions    : {candidate_rows}")
print(f"Categories            : {category_rows}")
print(f"Candidate decisions   : {len(candidate_decisions)} / {candidate_rows}")
print(f"Full-text reviews     : {completed_note_reviews} / {full_review_rows}")
print(
    "Category definitions : "
    f"{completed_category_definitions} / {category_rows}"
)

if repaired_cells:
    print("\nRepaired cells:")
    for repair in repaired_cells:
        print(
            f"- {repair['sheet']}!"
            f"{repair['cell']} "
            f"({repair['repair']})"
        )



**Populating and freezing operational category definitions**

In [ ]:
# 26. Populating and Freezing Operational Annotation Definitions

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

from openpyxl import load_workbook
from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side,
)

INPUT_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook_final.xlsx"
)

ANNOTATION_READY_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook_annotation_ready.xlsx"
)

DEFINITION_REPORT_FILE = (
    ANNOTATION_DIRECTORY
    / "category_definition_freeze_report.json"
)

if not INPUT_WORKBOOK_FILE.is_file():
    raise FileNotFoundError(
        "The audited workbook was not found:\n"
        f"{INPUT_WORKBOOK_FILE}"
    )


# Project-specific operational definitions

# These are annotation rules, not independent diagnostic criteria.
# The annotator must label explicit clinical documentation and must not infer
# diagnoses solely from laboratory results, medications, procedures or symptoms.


CATEGORY_DEFINITIONS = {
    "ACUTE_KIDNEY_INJURY": {
        "definition":
            "Explicit documentation of acute kidney injury, acute renal injury, "
            "acute renal failure or an equivalent acute deterioration in renal "
            "function during the relevant admission.",
        "include":
            "Include AKI, acute kidney injury, acute renal failure, acute renal "
            "insufficiency and clearly documented acute-on-chronic kidney injury. "
            "Retain historical, uncertain and negated mentions with the appropriate "
            "context labels.",
        "exclude":
            "Do not infer AKI from creatinine, oliguria, dialysis, fluid balance or "
            "medication changes alone. Exclude chronic kidney disease without an "
            "explicit acute component and end-stage renal disease without acute injury."
    },

    "ARDS": {
        "definition":
            "Explicit documentation of acute respiratory distress syndrome.",
        "include":
            "Include ARDS and acute respiratory distress syndrome. Include clinician "
            "statements describing ARDS as confirmed, suspected, resolved or historical, "
            "with the corresponding assertion and temporality labels.",
        "exclude":
            "Do not infer ARDS from hypoxaemia, bilateral infiltrates, mechanical "
            "ventilation, high PEEP or respiratory failure alone. Exclude generic "
            "respiratory distress without explicit ARDS documentation."
    },

    "ARRHYTHMIA": {
        "definition":
            "Explicit documentation of a clinically recognised abnormal cardiac rhythm.",
        "include":
            "Include atrial fibrillation, atrial flutter, supraventricular tachycardia, "
            "ventricular tachycardia, ventricular fibrillation, bradyarrhythmia, heart "
            "block and other explicitly named arrhythmias.",
        "exclude":
            "Do not infer an arrhythmia from an abnormal pulse, telemetry monitoring, "
            "electrolyte disturbance or anti-arrhythmic medication alone. Exclude isolated "
            "sinus tachycardia or sinus bradycardia unless the project protocol explicitly "
            "treats it as a complication."
    },

    "ASPIRATION": {
        "definition":
            "Explicit documentation of aspiration of gastric, oral, food or other "
            "material into the airway, including a clinically documented aspiration event.",
        "include":
            "Include aspiration, witnessed aspiration, aspiration event and aspiration "
            "pneumonitis. Aspiration pneumonia may also receive the PNEUMONIA category "
            "when pneumonia is explicitly documented.",
        "exclude":
            "Do not infer aspiration from dysphagia, vomiting, tube feeding, reduced "
            "consciousness or aspiration precautions alone. Exclude aspiration risk "
            "without an actual or suspected event."
    },

    "BLOODSTREAM_INFECTION": {
        "definition":
            "Explicit documentation of bloodstream infection, bacteraemia, fungaemia "
            "or catheter-related bloodstream infection.",
        "include":
            "Include bacteraemia, bloodstream infection, central-line-associated "
            "bloodstream infection, catheter-related bloodstream infection and fungaemia.",
        "exclude":
            "Do not infer infection from a positive blood culture alone unless the note "
            "interprets it clinically. Exclude contamination and isolated culture results "
            "described as contaminants."
    },

    "CARDIAC_ARREST": {
        "definition":
            "Explicit documentation of cardiac arrest or cardiopulmonary arrest requiring "
            "or prompting resuscitative action.",
        "include":
            "Include cardiac arrest, cardiopulmonary arrest, pulseless arrest, PEA arrest, "
            "asystolic arrest and ventricular-fibrillation arrest.",
        "exclude":
            "Do not infer arrest from CPR documentation without confirming context. "
            "Exclude respiratory arrest without cardiac arrest, syncope, hypotension and "
            "do-not-resuscitate discussions without an arrest event."
    },

    "CLOSTRIDIOIDES_DIFFICILE_INFECTION": {
        "definition":
            "Explicit documentation of Clostridioides difficile infection or clinically "
            "diagnosed C. difficile-associated diarrhoea or colitis.",
        "include":
            "Include C. difficile infection, C. diff infection, C. difficile colitis, "
            "C. diff colitis and pseudomembranous colitis when attributed to C. difficile.",
        "exclude":
            "Do not infer infection from diarrhoea, antibiotic exposure or a laboratory "
            "result alone. Exclude colonisation and negative testing."
    },

    "DEEP_VEIN_THROMBOSIS": {
        "definition":
            "Explicit documentation of thrombosis in a deep venous system.",
        "include":
            "Include DVT, deep vein thrombosis, upper-extremity DVT, lower-extremity DVT "
            "and catheter-associated deep venous thrombosis.",
        "exclude":
            "Exclude superficial thrombophlebitis, superficial venous thrombosis and "
            "arterial thrombosis. Do not infer DVT from swelling, anticoagulation or "
            "ultrasound ordering alone."
    },

    "DELIRIUM": {
        "definition":
            "Explicit documentation of delirium or an acute fluctuating confusional state "
            "identified clinically as delirium.",
        "include":
            "Include delirium, ICU delirium, hyperactive delirium, hypoactive delirium "
            "and acute confusional state when the note equates it with delirium.",
        "exclude":
            "Do not infer delirium from agitation, confusion, sedation, dementia or "
            "altered mental status alone. Exclude chronic cognitive impairment unless "
            "an acute delirium is explicitly documented."
    },

    "ENCEPHALOPATHY": {
        "definition":
            "Explicit documentation of encephalopathy, including toxic, metabolic, "
            "hypoxic, hepatic, uraemic or other clinically named encephalopathy.",
        "include":
            "Include acute encephalopathy, toxic-metabolic encephalopathy, hypoxic-ischaemic "
            "encephalopathy, hepatic encephalopathy and other explicit encephalopathy terms.",
        "exclude":
            "Do not infer encephalopathy from confusion, sedation, coma, altered mental "
            "status or abnormal laboratory values alone. Do not automatically convert "
            "delirium into encephalopathy."
    },

    "GASTROINTESTINAL_BLEEDING": {
        "definition":
            "Explicit documentation of bleeding originating from the gastrointestinal tract.",
        "include":
            "Include GI bleed, gastrointestinal haemorrhage, upper GI bleed, lower GI bleed, "
            "haematemesis, melaena or haematochezia when documented as gastrointestinal "
            "bleeding.",
        "exclude":
            "Do not infer GI bleeding from anaemia, transfusion, positive occult blood or "
            "endoscopy alone. Exclude non-gastrointestinal bleeding sources."
    },

    "MYOCARDIAL_INFARCTION": {
        "definition":
            "Explicit documentation of acute myocardial infarction or a named myocardial "
            "infarction subtype.",
        "include":
            "Include myocardial infarction, MI, acute MI, STEMI, NSTEMI and type 1 or type 2 "
            "myocardial infarction when explicitly documented.",
        "exclude":
            "Do not infer myocardial infarction from troponin elevation, ECG changes, chest "
            "pain or antiplatelet treatment alone. Exclude myocardial injury without an "
            "explicit infarction diagnosis."
    },

    "PLEURAL_EFFUSION": {
        "definition":
            "Explicit documentation of fluid accumulation in the pleural space.",
        "include":
            "Include pleural effusion, parapneumonic effusion, malignant pleural effusion "
            "and haemothorax only when the project decision is to treat it as an effusion "
            "and this is explicitly documented.",
        "exclude":
            "Do not infer pleural effusion from thoracentesis, chest-drain placement, "
            "dullness, reduced breath sounds or imaging language that is not interpreted "
            "as an effusion in the clinical note."
    },

    "PNEUMONIA": {
        "definition":
            "Explicit documentation of pneumonia or a clinically named pulmonary infection.",
        "include":
            "Include pneumonia, hospital-acquired pneumonia, ventilator-associated pneumonia, "
            "community-acquired pneumonia, aspiration pneumonia and explicitly documented "
            "bacterial, viral or fungal pneumonia.",
        "exclude":
            "Do not infer pneumonia from infiltrates, fever, sputum, hypoxaemia, antibiotics "
            "or respiratory cultures alone. Exclude pneumonitis unless the note also explicitly "
            "documents pneumonia."
    },

    "PNEUMOTHORAX": {
        "definition":
            "Explicit documentation of air in the pleural space causing a pneumothorax.",
        "include":
            "Include pneumothorax, tension pneumothorax, traumatic pneumothorax, spontaneous "
            "pneumothorax and iatrogenic pneumothorax.",
        "exclude":
            "Do not infer pneumothorax from chest-tube placement, trauma or ventilation alone. "
            "Exclude pneumomediastinum and subcutaneous emphysema unless pneumothorax is also "
            "explicitly documented."
    },

    "PRESSURE_INJURY": {
        "definition":
            "Explicit documentation of a pressure-related skin or soft-tissue injury.",
        "include":
            "Include pressure injury, pressure ulcer, pressure sore, decubitus ulcer and "
            "explicitly staged pressure injuries.",
        "exclude":
            "Exclude non-pressure wounds, moisture-associated skin damage, surgical wounds, "
            "skin tears and general redness unless identified as pressure-related."
    },

    "PULMONARY_EMBOLISM": {
        "definition":
            "Explicit documentation of embolic obstruction of the pulmonary arterial system.",
        "include":
            "Include pulmonary embolism, PE, saddle embolus, subsegmental PE and chronic "
            "thromboembolic disease when explicitly documented.",
        "exclude":
            "Do not infer PE from hypoxaemia, tachycardia, anticoagulation, D-dimer, CT "
            "angiography or DVT alone. Exclude fat or air embolism unless the annotation "
            "protocol explicitly includes these as pulmonary embolism."
    },

    "RESPIRATORY_FAILURE": {
        "definition":
            "Explicit documentation of acute, chronic or acute-on-chronic respiratory failure.",
        "include":
            "Include hypoxaemic respiratory failure, hypercapnic respiratory failure, acute "
            "respiratory failure and acute-on-chronic respiratory failure.",
        "exclude":
            "Do not infer respiratory failure solely from oxygen use, intubation, mechanical "
            "ventilation, hypoxaemia, hypercapnia or respiratory distress. Exclude respiratory "
            "insufficiency unless explicitly equated with respiratory failure."
    },

    "SEPSIS": {
        "definition":
            "Explicit documentation of sepsis as a clinical diagnosis.",
        "include":
            "Include sepsis, severe sepsis when explicitly documented, urosepsis when used by "
            "the clinician as a sepsis diagnosis, and sepsis attributed to a named infection.",
        "exclude":
            "Do not infer sepsis from infection, fever, tachycardia, leukocytosis, positive "
            "cultures, antibiotics or organ dysfunction alone. Exclude isolated SIRS without "
            "an explicit sepsis diagnosis."
    },

    "SEPTIC_SHOCK": {
        "definition":
            "Explicit documentation of septic shock.",
        "include":
            "Include septic shock and sepsis with shock when the clinician explicitly links "
            "the shock state to infection or sepsis.",
        "exclude":
            "Do not infer septic shock from vasopressor use, hypotension, lactate elevation "
            "or sepsis alone. Exclude cardiogenic, haemorrhagic, obstructive and other "
            "non-septic shock states."
    },

    "STROKE": {
        "definition":
            "Explicit documentation of an acute or historical cerebrovascular stroke.",
        "include":
            "Include ischaemic stroke, haemorrhagic stroke, cerebral infarction, intracerebral "
            "haemorrhage documented as stroke and cerebrovascular accident.",
        "exclude":
            "Exclude transient ischaemic attack unless stroke is also documented. Do not infer "
            "stroke from focal neurological deficits, imaging, thrombolysis or thrombectomy alone."
    },

    "URINARY_TRACT_INFECTION": {
        "definition":
            "Explicit documentation of infection involving the urinary tract.",
        "include":
            "Include urinary tract infection, UTI, catheter-associated UTI, cystitis and "
            "pyelonephritis when documented as a urinary infection.",
        "exclude":
            "Do not infer UTI from pyuria, bacteriuria, urine culture, urinary symptoms or "
            "antibiotic treatment alone. Exclude asymptomatic bacteriuria and colonisation."
    },

    "WOUND_INFECTION": {
        "definition":
            "Explicit documentation of infection involving a wound, incision, surgical site "
            "or traumatic tissue defect.",
        "include":
            "Include wound infection, surgical-site infection, infected incision, infected "
            "ulcer and explicitly documented postoperative wound infection.",
        "exclude":
            "Exclude uncomplicated wounds, cellulitis without a documented wound source, "
            "colonisation, drainage without infection and prophylactic antibiotic treatment."
    },
}

# Validating the dictionary against the workbook


workbook = load_workbook(
    INPUT_WORKBOOK_FILE,
    data_only=False,
)

if "Category_Definitions" not in workbook.sheetnames:
    raise ValueError(
        "Category_Definitions sheet is missing."
    )

definitions_ws = workbook["Category_Definitions"]

workbook_categories = []

for row_number in range(
    2,
    definitions_ws.max_row + 1,
):
    category = definitions_ws[
        f"A{row_number}"
    ].value

    if category is not None:
        workbook_categories.append(
            str(category).strip()
        )

dictionary_categories = set(
    CATEGORY_DEFINITIONS.keys()
)

workbook_category_set = set(
    workbook_categories
)

missing_definitions = sorted(
    workbook_category_set
    - dictionary_categories
)

extra_definitions = sorted(
    dictionary_categories
    - workbook_category_set
)

if missing_definitions or extra_definitions:
    raise ValueError(
        "Definition/category mismatch.\n"
        f"Missing definitions: {missing_definitions}\n"
        f"Unexpected definitions: {extra_definitions}"
    )

if len(workbook_categories) != 23:
    raise ValueError(
        "Expected 23 workbook categories, found "
        f"{len(workbook_categories)}."
    )


# Populating the Category_Definitions sheet

for row_number in range(
    2,
    definitions_ws.max_row + 1,
):
    category = str(
        definitions_ws[
            f"A{row_number}"
        ].value
    ).strip()

    definition_record = (
        CATEGORY_DEFINITIONS[category]
    )

    definitions_ws[
        f"B{row_number}"
    ] = definition_record["definition"]

    definitions_ws[
        f"C{row_number}"
    ] = definition_record["include"]

    definitions_ws[
        f"D{row_number}"
    ] = definition_record["exclude"]

# Adding metadata columns.
definitions_ws["E1"] = "definition_version"
definitions_ws["F1"] = "definition_status"

for row_number in range(
    2,
    definitions_ws.max_row + 1,
):
    definitions_ws[
        f"E{row_number}"
    ] = "1.0"

    definitions_ws[
        f"F{row_number}"
    ] = "FROZEN_FOR_PILOT"


# Formatting

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78",
)

header_font = Font(
    bold=True,
    color="FFFFFF",
)

frozen_fill = PatternFill(
    fill_type="solid",
    fgColor="E2F0D9",
)

thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

for cell in definitions_ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True,
    )

for row_number in range(
    2,
    definitions_ws.max_row + 1,
):
    for column_number in range(1, 7):
        cell = definitions_ws.cell(
            row=row_number,
            column=column_number,
        )

        cell.border = thin_border
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )

    for column_number in range(2, 7):
        definitions_ws.cell(
            row=row_number,
            column=column_number,
        ).fill = frozen_fill

definitions_ws.column_dimensions["A"].width = 38
definitions_ws.column_dimensions["B"].width = 68
definitions_ws.column_dimensions["C"].width = 78
definitions_ws.column_dimensions["D"].width = 78
definitions_ws.column_dimensions["E"].width = 20
definitions_ws.column_dimensions["F"].width = 22

definitions_ws.freeze_panes = "A2"
definitions_ws.auto_filter.ref = (
    f"A1:F{definitions_ws.max_row}"
)

# Adding global annotation principles to Instructions

instructions_ws = workbook["Instructions"]

principles = [
    (
        "Operational-definition status",
        "Category definitions version 1.0 are frozen for the pilot annotation. "
        "Any later amendment must be documented and versioned."
    ),
    (
        "Explicit-documentation rule",
        "Annotate conditions explicitly documented by the clinical author. "
        "Do not infer a complication solely from tests, medications, procedures, "
        "vital signs, laboratory values or imaging."
    ),
    (
        "Span rule",
        "Select the shortest text span that preserves the complete clinical concept. "
        "Do not include unrelated punctuation or surrounding prose."
    ),
    (
        "Assertion rule",
        "Assign AFFIRMED, NEGATED, UNCERTAIN, HYPOTHETICAL, HISTORICAL or FAMILY "
        "according to the local wording and clinical context."
    ),
    (
        "Temporality rule",
        "Use CURRENT for a condition active during the relevant admission, HISTORICAL "
        "for a prior or resolved condition, PLANNED_OR_FUTURE for projected events, "
        "and UNCLEAR when temporality cannot be resolved."
    ),
    (
        "Multiple-category rule",
        "A single sentence may contain more than one complication category. Annotate "
        "each explicit complication mention separately."
    ),
    (
        "No silent assumption",
        "When the category, assertion or temporality remains genuinely ambiguous, use "
        "REVIEW_REQUIRED and document the reason."
    ),
]

start_row = instructions_ws.max_row + 2

for offset, (
    heading,
    description,
) in enumerate(principles):

    row_number = start_row + offset

    instructions_ws.cell(
        row=row_number,
        column=1,
        value=heading,
    )

    instructions_ws.cell(
        row=row_number,
        column=2,
        value=description,
    )

    instructions_ws.cell(
        row=row_number,
        column=1,
    ).font = Font(bold=True)

    instructions_ws.cell(
        row=row_number,
        column=1,
    ).fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )

    for column_number in [1, 2]:
        cell = instructions_ws.cell(
            row=row_number,
            column=column_number,
        )

        cell.border = thin_border
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )

# Update Audit_Summary


if "Audit_Summary" not in workbook.sheetnames:
    raise ValueError(
        "Audit_Summary sheet is missing."
    )

audit_ws = workbook["Audit_Summary"]

# Updating values by matching labels rather than fixed rows.
audit_labels = {
    audit_ws[
        f"A{row_number}"
    ].value: row_number
    for row_number in range(
        1,
        audit_ws.max_row + 1,
    )
}

if "Complete category definitions" in audit_labels:
    row_number = audit_labels[
        "Complete category definitions"
    ]
    audit_ws[
        f"B{row_number}"
    ] = 23

if "Category definitions remaining" in audit_labels:
    row_number = audit_labels[
        "Category definitions remaining"
    ]
    audit_ws[
        f"B{row_number}"
    ] = 0

if "Notebook 07 readiness" in audit_labels:
    row_number = audit_labels[
        "Notebook 07 readiness"
    ]
    audit_ws[
        f"B{row_number}"
    ] = (
        "NOT READY — complete the 5-note pilot, then annotate "
        "and validate all 70 notes."
    )

# Creating a deterministic definition hash


definition_payload = {
    category: CATEGORY_DEFINITIONS[category]
    for category in sorted(
        CATEGORY_DEFINITIONS
    )
}

definition_json = json.dumps(
    definition_payload,
    sort_keys=True,
    ensure_ascii=False,
    separators=(",", ":"),
)

definition_sha256 = hashlib.sha256(
    definition_json.encode("utf-8")
).hexdigest()

# Adding definition metadata sheet.
if "Definition_Metadata" in workbook.sheetnames:
    del workbook["Definition_Metadata"]

metadata_ws = workbook.create_sheet(
    "Definition_Metadata",
    2,
)

metadata_rows = [
    ["field", "value"],
    ["definition_version", "1.0"],
    [
        "definition_status",
        "FROZEN_FOR_PILOT",
    ],
    [
        "definition_count",
        23,
    ],
    [
        "definition_sha256",
        definition_sha256,
    ],
    [
        "generated_timestamp_utc",
        datetime.now(
            timezone.utc
        ).isoformat(),
    ],
    [
        "scope",
        "Project-specific mention annotation rules; not diagnostic criteria.",
    ],
    [
        "amendment_policy",
        "Any change after pilot review must create a new version and hash.",
    ],
]

for row in metadata_rows:
    metadata_ws.append(row)

for cell in metadata_ws[1]:
    cell.fill = header_fill
    cell.font = header_font

metadata_ws.column_dimensions["A"].width = 30
metadata_ws.column_dimensions["B"].width = 105

for row in metadata_ws.iter_rows():
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True,
        )

# Save annotation-ready workbook.
workbook.save(
    ANNOTATION_READY_WORKBOOK_FILE
)


# Independent reload validation


validation_workbook = load_workbook(
    ANNOTATION_READY_WORKBOOK_FILE,
    read_only=True,
    data_only=False,
)

validation_definitions_ws = validation_workbook[
    "Category_Definitions"
]

completed_definitions = 0

for row_number in range(
    2,
    validation_definitions_ws.max_row + 1,
):
    values = [
        validation_definitions_ws[
            f"B{row_number}"
        ].value,
        validation_definitions_ws[
            f"C{row_number}"
        ].value,
        validation_definitions_ws[
            f"D{row_number}"
        ].value,
        validation_definitions_ws[
            f"E{row_number}"
        ].value,
        validation_definitions_ws[
            f"F{row_number}"
        ].value,
    ]

    if all(
        value is not None
        and str(value).strip()
        for value in values
    ):
        completed_definitions += 1

definition_validation = {
    "annotation_ready_workbook_exists":
        ANNOTATION_READY_WORKBOOK_FILE.is_file(),

    "all_23_definitions_complete":
        completed_definitions == 23,

    "definition_metadata_sheet_present":
        "Definition_Metadata"
        in validation_workbook.sheetnames,

    "audit_summary_sheet_present":
        "Audit_Summary"
        in validation_workbook.sheetnames,

    "candidate_review_sheet_present":
        "Candidate_Review"
        in validation_workbook.sheetnames,

    "full_text_review_sheet_present":
        "Full_Text_Review"
        in validation_workbook.sheetnames,
}

failed_checks = [
    check
    for check, passed
    in definition_validation.items()
    if not passed
]

if failed_checks:
    raise ValueError(
        "Definition freeze validation failed:\n- "
        + "\n- ".join(failed_checks)
    )

definition_report = {
    "definition_version": "1.0",
    "definition_status":
        "FROZEN_FOR_PILOT",
    "definition_count": 23,
    "definition_sha256":
        definition_sha256,
    "source_workbook":
        str(INPUT_WORKBOOK_FILE),
    "annotation_ready_workbook":
        str(ANNOTATION_READY_WORKBOOK_FILE),
    "generated_timestamp_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "validation": {
        key: bool(value)
        for key, value
        in definition_validation.items()
    },
    "next_stage":
        "Five-note pilot manual annotation",
}

with open(
    DEFINITION_REPORT_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        definition_report,
        file,
        indent=2,
    )

print("-" * 90)
print("CATEGORY DEFINITIONS POPULATED AND FROZEN")
print("-" * 90)

print(
    f"Input workbook       : "
    f"{INPUT_WORKBOOK_FILE}"
)
print(
    f"Annotation workbook  : "
    f"{ANNOTATION_READY_WORKBOOK_FILE}"
)
print(
    f"Definition report    : "
    f"{DEFINITION_REPORT_FILE}"
)
print(
    f"Definitions complete : "
    f"{completed_definitions} / 23"
)
print(
    f"Definition version   : 1.0"
)
print(
    f"Definition status    : "
    f"FROZEN_FOR_PILOT"
)
print(
    f"Definition SHA-256   : "
    f"{definition_sha256}"
)
print(
    f"Validation checks    : "
    f"{len(definition_validation)} passed"
)

print(
    "\nNext stage: conduct a controlled "
    "five-note pilot annotation."
)

**Selection and documention of the five-note pilot**

In [ ]:
# 27. Final Validation

#
# This cell validates the completed five-note pilot workbook.
# It does NOT resample notes, recreate the pilot, or overwrite annotations.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd
from openpyxl import load_workbook
from IPython.display import display



# 1. File paths


COMPLETED_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook_notebook06_complete.xlsx"
)

COMPLETION_REPORT_FILE = (
    ANNOTATION_DIRECTORY
    / "notebook_06_completion_report.json"
)

if not COMPLETED_WORKBOOK_FILE.is_file():
    raise FileNotFoundError(
        "The completed Notebook 06 workbook was not found:\n"
        f"{COMPLETED_WORKBOOK_FILE}\n\n"
        "Upload the following file to the gold_standard_annotation folder:\n"
        "gold_standard_candidate_assisted_workbook_notebook06_complete.xlsx"
    )



# 2. Controlled five-note pilot identifiers


PILOT_NOTE_IDS = set(
    pd.read_excel(
        COMPLETED_WORKBOOK_FILE,
        sheet_name="Pilot_Selection",
        dtype="string",
    )["note_id"].dropna().astype("string").str.strip()
)

EXPECTED_PILOT_NOTES = 5
EXPECTED_PILOT_CANDIDATES = 34
EXPECTED_MISSED_MENTIONS = 10


# 3. Workbook structural validation


workbook = load_workbook(
    COMPLETED_WORKBOOK_FILE,
    data_only=False,
)

required_sheets = {
    "Instructions",
    "Audit_Summary",
    "Definition_Metadata",
    "Note_Register",
    "Candidate_Review",
    "Full_Text_Review",
    "Mention_Annotations",
    "Category_Definitions",
    "Validation_Lists",
    "Pilot_Selection",
}

missing_sheets = sorted(
    required_sheets - set(workbook.sheetnames)
)

if missing_sheets:
    raise ValueError(
        "The completed workbook is missing required sheets:\n- "
        + "\n- ".join(missing_sheets)
    )



# 4. Loading completed workbook tables


note_register_df = pd.read_excel(
    COMPLETED_WORKBOOK_FILE,
    sheet_name="Note_Register",
    dtype="string",
)

candidate_review_df = pd.read_excel(
    COMPLETED_WORKBOOK_FILE,
    sheet_name="Candidate_Review",
    dtype="string",
)

full_text_review_df = pd.read_excel(
    COMPLETED_WORKBOOK_FILE,
    sheet_name="Full_Text_Review",
    dtype="string",
)

mention_annotations_df = pd.read_excel(
    COMPLETED_WORKBOOK_FILE,
    sheet_name="Mention_Annotations",
    dtype="string",
)

category_definitions_df = pd.read_excel(
    COMPLETED_WORKBOOK_FILE,
    sheet_name="Category_Definitions",
    dtype="string",
)

pilot_selection_df = pd.read_excel(
    COMPLETED_WORKBOOK_FILE,
    sheet_name="Pilot_Selection",
    dtype="string",
)



# 5. Normalising identifiers

def normalise_identifier(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


for dataframe in [
    note_register_df,
    candidate_review_df,
    full_text_review_df,
    mention_annotations_df,
    pilot_selection_df,
]:
    if "note_id" in dataframe.columns:
        dataframe["note_id"] = normalise_identifier(
            dataframe["note_id"]
        )



# 6. Isolating pilot records


pilot_note_register_df = note_register_df[
    note_register_df["note_id"].isin(PILOT_NOTE_IDS)
].copy()

pilot_candidate_df = candidate_review_df[
    candidate_review_df["note_id"].isin(PILOT_NOTE_IDS)
].copy()

pilot_full_text_df = full_text_review_df[
    full_text_review_df["note_id"].isin(PILOT_NOTE_IDS)
].copy()

pilot_missed_mentions_df = mention_annotations_df[
    mention_annotations_df["note_id"].isin(PILOT_NOTE_IDS)
].copy()

pilot_selection_subset_df = pilot_selection_df[
    pilot_selection_df["note_id"].isin(PILOT_NOTE_IDS)
].copy()



# 7. Calculating pilot completion statistics


valid_decisions = {
    "KEEP",
    "MODIFY",
    "DELETE",
    "REVIEW_REQUIRED",
}

pilot_candidate_df["reviewer_decision"] = (
    pilot_candidate_df["reviewer_decision"]
    .astype("string")
    .str.strip()
    .str.upper()
)

decision_counts = (
    pilot_candidate_df["reviewer_decision"]
    .value_counts(dropna=False)
    .to_dict()
)

completed_candidate_decisions = int(
    pilot_candidate_df["reviewer_decision"]
    .isin(valid_decisions)
    .sum()
)

keep_count = int(
    (
        pilot_candidate_df["reviewer_decision"]
        == "KEEP"
    ).sum()
)

modify_count = int(
    (
        pilot_candidate_df["reviewer_decision"]
        == "MODIFY"
    ).sum()
)

delete_count = int(
    (
        pilot_candidate_df["reviewer_decision"]
        == "DELETE"
    ).sum()
)

review_required_count = int(
    (
        pilot_candidate_df["reviewer_decision"]
        == "REVIEW_REQUIRED"
    ).sum()
)

completed_note_register_count = int(
    (
        pilot_note_register_df["annotation_status"]
        .astype("string")
        .str.strip()
        .str.upper()
        == "COMPLETED"
    ).sum()
)

completed_full_text_review_count = int(
    (
        pilot_full_text_df["note_annotation_complete"]
        .astype("string")
        .str.strip()
        .str.upper()
        == "YES"
    ).sum()
)

candidate_review_complete_count = int(
    (
        pilot_full_text_df["candidate_review_complete"]
        .astype("string")
        .str.strip()
        .str.upper()
        == "YES"
    ).sum()
)

original_text_reviewed_count = int(
    (
        pilot_full_text_df["full_original_text_reviewed"]
        .astype("string")
        .str.strip()
        .str.upper()
        == "YES"
    ).sum()
)

missed_mentions_added = int(
    pilot_missed_mentions_df["annotation_id"]
    .notna()
    .sum()
)

frozen_definition_count = int(
    (
        category_definitions_df["definition_status"]
        .astype("string")
        .str.strip()
        .str.upper()
        == "FROZEN_FOR_PILOT"
    ).sum()
)



# 8. Final validation checks


validation_checks = {
    "completed_workbook_exists":
        COMPLETED_WORKBOOK_FILE.is_file(),

    "all_required_sheets_present":
        len(missing_sheets) == 0,

    "sample_contains_70_notes":
        len(note_register_df) == 70,

    "candidate_table_contains_380_rows":
        len(candidate_review_df) == 380,

    "pilot_contains_5_notes":
        len(pilot_note_register_df) == EXPECTED_PILOT_NOTES,

    "pilot_selection_contains_5_notes":
        len(pilot_selection_subset_df) == EXPECTED_PILOT_NOTES,

    "pilot_contains_34_candidates":
        len(pilot_candidate_df) == EXPECTED_PILOT_CANDIDATES,

    "all_34_candidate_decisions_completed":
        completed_candidate_decisions
        == EXPECTED_PILOT_CANDIDATES,

    "all_5_note_register_records_completed":
        completed_note_register_count
        == EXPECTED_PILOT_NOTES,

    "all_5_candidate_reviews_completed":
        candidate_review_complete_count
        == EXPECTED_PILOT_NOTES,

    "all_5_original_texts_reviewed":
        original_text_reviewed_count
        == EXPECTED_PILOT_NOTES,

    "all_5_note_annotations_completed":
        completed_full_text_review_count
        == EXPECTED_PILOT_NOTES,

    "10_missed_mentions_recorded":
        missed_mentions_added
        == EXPECTED_MISSED_MENTIONS,

    "all_23_category_definitions_frozen":
        frozen_definition_count == 23,

    "no_unresolved_pilot_reviews":
        review_required_count == 0,
}

failed_checks = [
    check_name
    for check_name, passed
    in validation_checks.items()
    if not passed
]

if failed_checks:
    print("=" * 72)
    print("NOTEBOOK 06 FINAL VALIDATION: FAILED")
    print("=" * 72)

    for check_name, passed in validation_checks.items():
        symbol = "PASS" if passed else "FAIL"
        print(f"{symbol:4} | {check_name}")

    raise ValueError(
        "\nNotebook 06 cannot be closed because the following "
        "validation checks failed:\n- "
        + "\n- ".join(failed_checks)
    )



# 9. Generating completion report


workbook_sha256 = hashlib.sha256(
    COMPLETED_WORKBOOK_FILE.read_bytes()
).hexdigest()

completion_report = {
    "notebook": "06",
    "notebook_title": (
        "Gold-Standard Clinical Complication Annotation"
    ),
    "status": "COMPLETE",
    "pilot_status": "COMPLETED_MODEL_ASSISTED",
    "completion_timestamp_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "completed_workbook": str(
        COMPLETED_WORKBOOK_FILE
    ),
    "completed_workbook_sha256": workbook_sha256,
    "sample_notes": int(len(note_register_df)),
    "total_candidate_mentions": int(
        len(candidate_review_df)
    ),
    "pilot_notes": int(
        len(pilot_note_register_df)
    ),
    "pilot_candidate_mentions": int(
        len(pilot_candidate_df)
    ),
    "pilot_candidate_decisions_completed": (
        completed_candidate_decisions
    ),
    "pilot_decision_counts": {
        "KEEP": keep_count,
        "MODIFY": modify_count,
        "DELETE": delete_count,
        "REVIEW_REQUIRED": review_required_count,
    },
    "pilot_full_text_reviews_completed": (
        completed_full_text_review_count
    ),
    "pilot_missed_mentions_added": (
        missed_mentions_added
    ),
    "category_definitions_frozen": (
        frozen_definition_count
    ),
    "validation_checks": validation_checks,
    "validation_checks_passed": int(
        sum(validation_checks.values())
    ),
    "validation_checks_failed": int(
        len(failed_checks)
    ),
    "human_verification_required": True,
    "research_note": (
        "The five-note pilot was completed using "
        "model-assisted annotation. Independent human "
        "verification is recommended before the annotations "
        "are described as a definitive clinical gold standard."
    ),
    "next_stage": (
        "Proceed to Notebook 07 for evaluation and analysis."
    ),
}

with open(
    COMPLETION_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        completion_report,
        report_file,
        indent=2,
        ensure_ascii=False,
    )



# 10. Displaying final pilot summary


pilot_summary_df = pd.DataFrame(
    {
        "Metric": [
            "Pilot notes completed",
            "Pilot candidate mentions",
            "Candidate decisions completed",
            "KEEP decisions",
            "MODIFY decisions",
            "DELETE decisions",
            "REVIEW_REQUIRED decisions",
            "Full-text reviews completed",
            "Missed mentions added",
            "Frozen category definitions",
            "Validation checks passed",
        ],
        "Value": [
            len(pilot_note_register_df),
            len(pilot_candidate_df),
            completed_candidate_decisions,
            keep_count,
            modify_count,
            delete_count,
            review_required_count,
            completed_full_text_review_count,
            missed_mentions_added,
            frozen_definition_count,
            sum(validation_checks.values()),
        ],
    }
)

display(pilot_summary_df)


# 11. Final status


print("\n" + "-" * 72)
print("NOTEBOOK 06 — FINAL COMPLETION STATUS")
print("-" * 72)

print(f"\nCompleted workbook:\n{COMPLETED_WORKBOOK_FILE}")

print(
    f"\nWorkbook SHA-256:\n{workbook_sha256}"
)

print(
    f"\nCompletion report saved to:\n"
    f"{COMPLETION_REPORT_FILE}"
)

print("\nPilot status: COMPLETED_MODEL_ASSISTED")
print("Notebook 06 status: COMPLETE")

print(
    "\nAll final validation checks passed successfully."
)

print(
    "\nThe five-note pilot contains:"
    f"\n- {len(pilot_note_register_df)} completed notes"
    f"\n- {len(pilot_candidate_df)} reviewed candidate mentions"
    f"\n- {missed_mentions_added} manually added missed mentions"
)

print(
    "\nResearch limitation: the pilot was completed using "
    "model-assisted annotation. Human verification is "
    "recommended before describing it as a definitive "
    "clinical gold standard."
)

**Full 70-Note Gold-Standard Corpus Validation and Closure**

In [ ]:
# 28. Full 70-Note Gold-Standard Corpus Validation and Closure

#
# This cell runs after the five-note pilot validation.
# It validates the completed annotation corpus across all 70 notes.
# It does not recreate or overwrite any annotation records.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd
from openpyxl import load_workbook
from IPython.display import display



# 1. File paths


FULL_CORPUS_WORKBOOK_FILE = (
    ANNOTATION_DIRECTORY
    / "gold_standard_candidate_assisted_workbook_notebook06_complete_70_model_.xlsx"
)

FULL_CORPUS_REPORT_FILE = (
    ANNOTATION_DIRECTORY
    / "notebook_06_full_corpus_completion_report.json"
)

if not FULL_CORPUS_WORKBOOK_FILE.is_file():
    raise FileNotFoundError(
        "The completed 70-note workbook was not found:\n"
        f"{FULL_CORPUS_WORKBOOK_FILE}"
    )



# 2. Expected corpus sizes


EXPECTED_TOTAL_NOTES = 70
EXPECTED_PILOT_NOTES = 5
EXPECTED_REMAINING_NOTES = 65


# 3. Workbook structural validation


workbook = load_workbook(
    FULL_CORPUS_WORKBOOK_FILE,
    data_only=False,
)

required_sheets = {
    "Instructions",
    "Audit_Summary",
    "Definition_Metadata",
    "Note_Register",
    "Candidate_Review",
    "Full_Text_Review",
    "Mention_Annotations",
    "Category_Definitions",
    "Validation_Lists",
    "Pilot_Selection",
}

missing_sheets = sorted(
    required_sheets - set(workbook.sheetnames)
)

if missing_sheets:
    raise ValueError(
        "The workbook is missing required sheets:\n- "
        + "\n- ".join(missing_sheets)
    )



# 4. Loading full-corpus tables


note_register_df = pd.read_excel(
    FULL_CORPUS_WORKBOOK_FILE,
    sheet_name="Note_Register",
    dtype="string",
)

candidate_review_df = pd.read_excel(
    FULL_CORPUS_WORKBOOK_FILE,
    sheet_name="Candidate_Review",
    dtype="string",
)

full_text_review_df = pd.read_excel(
    FULL_CORPUS_WORKBOOK_FILE,
    sheet_name="Full_Text_Review",
    dtype="string",
)

mention_annotations_df = pd.read_excel(
    FULL_CORPUS_WORKBOOK_FILE,
    sheet_name="Mention_Annotations",
    dtype="string",
)

category_definitions_df = pd.read_excel(
    FULL_CORPUS_WORKBOOK_FILE,
    sheet_name="Category_Definitions",
    dtype="string",
)

pilot_selection_df = pd.read_excel(
    FULL_CORPUS_WORKBOOK_FILE,
    sheet_name="Pilot_Selection",
    dtype="string",
)



# 5. Normalise identifiers and status fields


def normalise_identifier(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


for dataframe in [
    note_register_df,
    candidate_review_df,
    full_text_review_df,
    mention_annotations_df,
    pilot_selection_df,
]:
    if "note_id" in dataframe.columns:
        dataframe["note_id"] = normalise_identifier(
            dataframe["note_id"]
        )


candidate_review_df["reviewer_decision"] = (
    candidate_review_df["reviewer_decision"]
    .astype("string")
    .str.strip()
    .str.upper()
)

note_register_df["annotation_status"] = (
    note_register_df["annotation_status"]
    .astype("string")
    .str.strip()
    .str.upper()
)

for column in [
    "candidate_review_complete",
    "full_original_text_reviewed",
    "note_annotation_complete",
]:
    full_text_review_df[column] = (
        full_text_review_df[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )



# 6. Calculating complete-corpus statistics


valid_final_decisions = {
    "KEEP",
    "MODIFY",
    "DELETE",
}

completed_note_count = int(
    (
        note_register_df["annotation_status"]
        == "COMPLETED"
    ).sum()
)

remaining_note_count = max(
    completed_note_count - EXPECTED_PILOT_NOTES,
    0,
)

completed_candidate_decisions = int(
    candidate_review_df["reviewer_decision"]
    .isin(valid_final_decisions)
    .sum()
)

keep_count = int(
    (
        candidate_review_df["reviewer_decision"]
        == "KEEP"
    ).sum()
)

modify_count = int(
    (
        candidate_review_df["reviewer_decision"]
        == "MODIFY"
    ).sum()
)

delete_count = int(
    (
        candidate_review_df["reviewer_decision"]
        == "DELETE"
    ).sum()
)

review_required_count = int(
    (
        candidate_review_df["reviewer_decision"]
        == "REVIEW_REQUIRED"
    ).sum()
)

blank_decision_count = int(
    candidate_review_df["reviewer_decision"]
    .isna()
    .sum()
)

candidate_review_complete_count = int(
    (
        full_text_review_df["candidate_review_complete"]
        == "YES"
    ).sum()
)

original_text_reviewed_count = int(
    (
        full_text_review_df["full_original_text_reviewed"]
        == "YES"
    ).sum()
)

full_text_completed_count = int(
    (
        full_text_review_df["note_annotation_complete"]
        == "YES"
    ).sum()
)

annotation_record_count = int(
    mention_annotations_df["annotation_id"]
    .notna()
    .sum()
)

frozen_definition_count = int(
    category_definitions_df["definition_status"]
    .astype("string")
    .str.strip()
    .str.upper()
    .isin({"FROZEN_FOR_PILOT", "FROZEN"})
    .sum()
)



# 7. Full-corpus validation checks


validation_checks = {
    "completed_workbook_exists":
        FULL_CORPUS_WORKBOOK_FILE.is_file(),

    "all_required_sheets_present":
        len(missing_sheets) == 0,

    "corpus_contains_70_notes":
        len(note_register_df) == EXPECTED_TOTAL_NOTES,

    "all_70_note_register_records_completed":
        completed_note_count == EXPECTED_TOTAL_NOTES,

    "remaining_65_notes_completed":
        remaining_note_count == EXPECTED_REMAINING_NOTES,

    "all_candidate_decisions_completed":
        completed_candidate_decisions
        == len(candidate_review_df),

    "no_review_required_decisions":
        review_required_count == 0,

    "no_blank_candidate_decisions":
        blank_decision_count == 0,

    "all_70_candidate_reviews_completed":
        candidate_review_complete_count
        == EXPECTED_TOTAL_NOTES,

    "all_70_original_texts_reviewed":
        original_text_reviewed_count
        == EXPECTED_TOTAL_NOTES,

    "all_70_note_annotations_completed":
        full_text_completed_count
        == EXPECTED_TOTAL_NOTES,

    "mention_annotations_present":
        annotation_record_count > 0,

    "category_definitions_available":
        frozen_definition_count > 0,
}

failed_checks = [
    check_name
    for check_name, passed in validation_checks.items()
    if not passed
]

if failed_checks:
    print("=" * 80)
    print("FULL 70-NOTE CORPUS VALIDATION: FAILED")
    print("=" * 80)

    for check_name, passed in validation_checks.items():
        result = "PASS" if passed else "FAIL"
        print(f"{result:4} | {check_name}")

    raise ValueError(
        "\nThe 70-note corpus cannot be marked complete because "
        "the following checks failed:\n- "
        + "\n- ".join(failed_checks)
    )



# 8. Workbook checksum and completion report


workbook_sha256 = hashlib.sha256(
    FULL_CORPUS_WORKBOOK_FILE.read_bytes()
).hexdigest()

completion_report = {
    "notebook": "06",
    "notebook_title":
        "Gold-Standard Clinical Complication Annotation",

    "status": "COMPLETE",
    "annotation_status":
        "COMPLETED_MODEL_ASSISTED_RESEARCHER_REVIEW",

    "completion_timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "completed_workbook":
        str(FULL_CORPUS_WORKBOOK_FILE),

    "completed_workbook_sha256":
        workbook_sha256,

    "total_notes":
        int(len(note_register_df)),

    "pilot_notes":
        EXPECTED_PILOT_NOTES,

    "remaining_notes":
        EXPECTED_REMAINING_NOTES,

    "completed_notes":
        completed_note_count,

    "total_candidate_mentions":
        int(len(candidate_review_df)),

    "candidate_decisions_completed":
        completed_candidate_decisions,

    "decision_counts": {
        "KEEP": keep_count,
        "MODIFY": modify_count,
        "DELETE": delete_count,
        "REVIEW_REQUIRED": review_required_count,
    },

    "candidate_reviews_completed":
        candidate_review_complete_count,

    "full_original_texts_reviewed":
        original_text_reviewed_count,

    "full_text_annotations_completed":
        full_text_completed_count,

    "mention_annotation_records":
        annotation_record_count,

    "category_definitions_frozen":
        frozen_definition_count,

    "validation_checks":
        validation_checks,

    "validation_checks_passed":
        int(sum(validation_checks.values())),

    "validation_checks_failed":
        int(len(failed_checks)),

    "human_verification_required": True,

    "research_note": (
        "The 70-note corpus was prepared through a "
        "model-assisted annotation workflow with researcher "
        "review. The annotation provenance and limitations "
        "must be reported transparently."
    ),

    "next_stage":
        "Proceed to Notebook 07 for system evaluation.",
}

with open(
    FULL_CORPUS_REPORT_FILE,
    "w",
    encoding="utf-8",
) as report_file:
    json.dump(
        completion_report,
        report_file,
        indent=2,
        ensure_ascii=False,
    )



# 9. Display full-corpus summary


full_corpus_summary_df = pd.DataFrame(
    {
        "Metric": [
            "Pilot notes completed",
            "Remaining notes completed",
            "Total annotated notes",
            "Total candidate mentions",
            "Candidate decisions completed",
            "KEEP decisions",
            "MODIFY decisions",
            "DELETE decisions",
            "REVIEW_REQUIRED decisions",
            "Candidate reviews completed",
            "Full original texts reviewed",
            "Full-text annotations completed",
            "Mention annotation records",
            "Frozen category definitions",
            "Validation checks passed",
        ],
        "Value": [
            EXPECTED_PILOT_NOTES,
            remaining_note_count,
            completed_note_count,
            len(candidate_review_df),
            completed_candidate_decisions,
            keep_count,
            modify_count,
            delete_count,
            review_required_count,
            candidate_review_complete_count,
            original_text_reviewed_count,
            full_text_completed_count,
            annotation_record_count,
            frozen_definition_count,
            sum(validation_checks.values()),
        ],
    }
)

display(full_corpus_summary_df)



# 10. Final status


print("\n" + "-" * 80)
print("FULL CORPUS COMPLETION STATUS")
print("-" * 80)

print(
    f"\nCompleted workbook:\n"
    f"{FULL_CORPUS_WORKBOOK_FILE}"
)

print(
    f"\nWorkbook SHA-256:\n"
    f"{workbook_sha256}"
)

print(
    f"\nCompletion report saved to:\n"
    f"{FULL_CORPUS_REPORT_FILE}"
)

print(
    "\nPilot annotation status: COMPLETE "
    f"({EXPECTED_PILOT_NOTES}/{EXPECTED_PILOT_NOTES})"
)

print(
    "Remaining annotation status: COMPLETE "
    f"({remaining_note_count}/{EXPECTED_REMAINING_NOTES})"
)

print(
    "Full annotation corpus: COMPLETE "
    f"({completed_note_count}/{EXPECTED_TOTAL_NOTES})"
)

print("Notebook 06 status: COMPLETE")
print("Next stage: Notebook 07 — Evaluation")

print(
    "\nAll full-corpus validation checks passed successfully."
)

print(
    "\nThe completed annotation corpus contains:"
    f"\n- {completed_note_count} annotated discharge summaries"
    f"\n- {completed_candidate_decisions} reviewed candidate mentions"
    f"\n- {annotation_record_count} mention-annotation records"
    f"\n- {review_required_count} unresolved reviews"
)